# Prompt Engineering

## Preparations and Settings

This document explains the prompt engineering (Step 3) and prompt evaluation (Step 4) experiments after the RAG agent was built to define the best prompts to use for the agent. The following cells will connect us to `Qwen3 Next 80B` and `GPT-OSS 120B` for comparisons









In [1]:
import os
from together import Together

together_client = Together(api_key=os.getenv("TOGETHER_API_KEY"))

def qwen_completion(system_prompt: str, user_prompt: str, temperature: float = 0.7) -> str:
    """Call Qwen3 Next 80B A3b Instruct via Together."""
    resp = together_client.chat.completions.create(
        model="Qwen/Qwen3-Next-80B-A3B-Instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


In [3]:
def gpt_oss_completion(system_prompt: str, user_prompt: str, temperature: float = 0.7) -> str:
    """Call OpenAI GPT-OSS 120B via Together."""
    resp = together_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


For the next steps **docker** is needed to embed our questions and to figure out which of these LLMs is better at being our proposify agent.


In [ ]:
# install docker Python package into this venv (optional)
%python -m pip install docker


In [ ]:
# run from the rag-chatbot folder
%docker compose up -d chroma ollama rag-chatbot


In [42]:
import docker
client = docker.from_env()
print(client.containers.list())


[<Container: 47164d5f791d>, <Container: 58a4076f9799>, <Container: e59b5331d958>]


In [46]:
import os
from dotenv import load_dotenv

load_dotenv("rag-chatbot/.env")

# Override container-style hosts with host ports
os.environ["CHROMA_HOST"] = "localhost"
os.environ["CHROMA_PORT"] = "8000"
os.environ["OLLAMA_BASE"] = "http://localhost:11434"
os.environ["EMBEDDING_URL"] = "http://localhost:11434/api/embeddings"


This is just to render the markdown answeres.

In [5]:
from IPython.display import Markdown, display

def show_markdown(text: str, title: str | None = None):
    if title:
        display(Markdown(f"### {title}\n\n{text}"))
    else:
        display(Markdown(text))


## Persona and Critics

To figure out if different personalities will give us an advantage or not we are going to compare answeres with and without persona setting for both of our LLMs.
We will first focus on a situation when the Student is trying to come up with a reseach question.


In [19]:
feedback_question = """
Please give me feedback on this research question for my bachelor thesis:

"How does the use of AI writing assistants (such as ChatGPT) influence the quality
and originality of bachelor students' academic essays at a Swiss university?"

Comment briefly on clarity, feasibility, and how I could improve it.
"""

qwen_text = qwen_completion(
    system_prompt=(
        "You are a helpful academic writing assistant. "
    ),
    user_prompt=feedback_question,
    temperature=0.5,
)

show_markdown(qwen_text, "Qwen – feedback")




### Qwen – feedback

Your research question is **clear** and **relevant**, addressing a timely issue in academic integrity and pedagogy. It specifies the tool (AI writing assistants like ChatGPT), the population (bachelor students), the context (Swiss university), and the outcomes of interest (quality and originality)—all strong elements.

**Feasibility**: Mostly feasible, but could be improved. Measuring “quality” and “originality” objectively is challenging. You’ll need clear, operational definitions (e.g., rubrics for quality; Turnitin/AI detection tools for originality) and a robust methodology (e.g., controlled comparison, surveys, or analysis of drafts with/without AI use).

**Suggestions for improvement**:
1. **Narrow the scope**: “Quality” is broad. Consider specifying dimensions (e.g., coherence, argumentation, grammar, depth of analysis).
2. **Add methodological hint**: E.g., “...as assessed through instructor evaluations and AI-detection tools?”
3. **Consider comparison**: Are you comparing essays written with vs. without AI? If so, make that explicit.

**Improved version**:
> “How does the use of AI writing assistants (such as ChatGPT) affect the argumentative depth, coherence, and originality of bachelor students’ academic essays at a Swiss university, as evaluated by instructors and AI-detection software?”

This version enhances precision, feasibility, and measurability—key for a bachelor thesis. Good start!

In [18]:
feedback_question = """
Please give me feedback on this research question for my bachelor thesis:

"How does the use of AI writing assistants (such as ChatGPT) influence the quality
and originality of bachelor students' academic essays at a Swiss university?"

Comment briefly on clarity, feasibility, and how I could improve it.
"""

gpt_oss_text= gpt_oss_completion(
    system_prompt=(
        "You are a helpful academic writing assistant. "
    ),
    user_prompt=feedback_question,
    temperature=0.5,
)

show_markdown(gpt_oss_text, "GPT‑OSS 120B – feedback")



### GPT‑OSS 120B – feedback

**Feedback on the research question**

> *“How does the use of AI writing assistants (such as ChatGPT) influence the quality and originality of bachelor students' academic essays at a Swiss university?”*

| Dimension | What works well | Potential issues | How to improve |
|-----------|----------------|------------------|----------------|
| **Clarity** | • The question is phrased in plain language. <br>• It identifies the two main outcomes you want to examine (quality & originality) and the population (bachelor students at a Swiss university). | • “Quality” and “originality” are broad, ambiguous terms. <br>• “Influence” can be interpreted as a causal effect, a correlation, or a perceived impact. <br>• “AI writing assistants (such as ChatGPT)” lumps together a whole class of tools that may differ in functionality. | • Define the constructs you will measure (e.g., “quality = rubric‑based score on argumentation, structure, and referencing”; “originality = similarity index from plagiarism‑detection software”). <br>• Specify the direction of inquiry (causal effect vs. association). <br>• If you plan to focus on one tool, name it; if you want to compare tools, state that explicitly. |
| **Feasibility** | • The topic is timely and there should be a readily available pool of essays. <br>• Access to a single university makes data collection manageable. | • Obtaining a sufficiently large, ethically cleared sample of essays (especially pre‑ and post‑AI‑use) can be time‑consuming. <br>• Measuring “influence” requires a research design that can isolate the AI variable (e.g., experimental vs. observational). <br>• Originality detection tools may flag AI‑generated text as “non‑original” even when it is not plagiarised, complicating interpretation. | • Choose a realistic design (e.g., a quasi‑experimental study where one cohort is allowed to use ChatGPT and another is not). <br>• Pilot the rubric and similarity‑index thresholds with a small set of essays to ensure reliability. <br>• Secure ethics approval early and consider anonymising essays to protect student privacy. |
| **Improvement suggestions** | • The core idea is solid and aligns with current scholarly debates. | • As written, the question is too broad for a bachelor‑level thesis (covers two outcomes, a whole class of tools, and an entire university). | **1. Narrow the scope** – focus on a single tool or a single outcome, or limit the study to a specific faculty or course. <br>**2. Make the construct operational** – embed the measurement approach in the question. <br>**3. Clarify the causal stance** – indicate whether you will test “the effect of” (experimental) or “the relationship between” (correlational). <br>**4. Add a temporal or comparative element** – e.g., “before vs. after the introduction of ChatGPT in 2023.” |

---

### A revised version (example)

> **“What effect does the optional use of ChatGPT have on (a) rubric‑based quality scores and (b) Turnitin similarity indices of bachelor‑level essays in the Business Administration program at the University of Zurich?"**

**Why this works better**

1. **Specific tool** – “ChatGPT” rather than “AI writing assistants.”  
2. **Clear outcomes** – quality measured by a validated rubric; originality measured by a similarity index.  
3. **Defined context** – one program at one university, making data collection realistic.  
4. **Causal wording** – “What effect does … have” signals an experimental or quasi‑experimental design.  

---

### Quick checklist for your final question

- [ ] **Constructs defined** – Have you specified how you will assess “quality” and “originality”?  
- [ ] **Population bounded** – Which faculty, year, or course will you sample?  
- [ ] **Tool clarified** – One AI assistant or a comparison?  
- [ ] **Design hinted** – Does the wording reflect the method you can actually implement?  
- [ ] **Ethical feasibility** – Can you obtain essays and consent without breaching privacy?  

Feel free to adapt the suggestions to fit the resources and timeline you have. Good luck with your thesis!

<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    Each of the two cells was executed four times. The results indicate that
    <code>GPT-OSS 120B</code> generally produces longer outputs with tabular structure,
    while <code>Qwen3 80B</code> provides shorter and more compact responses.
  </div>
</div>


Now lets give feedback with mode seetings and persona seetings. First we prepare the prompts.

In [9]:
MODE_INSTR = {
    "Research question helper": (
        "Help the student define or refine a precise, feasible research question; "
        "ask clarifying questions; propose concrete next steps."
    ),
    "Proposal refinement assistant": (
        "Guide refinement of structure, methods, datasets, ethics, and BFH compliance; "
        "provide step-by-step edits and a short improvement plan."
    ),
}

PERSONA_MAP = {
    "Supervisor": {
        "temp": 0.1,
        "instr": (
            "Strict critique. Identify weaknesses, risks, missing operationalization, "
            "measurement issues, and BFH compliance gaps. Add a short self-reflection "
            "checklist at the end."
        ),
    },
    "Helper": {
        "temp": 0.5,
        "instr": (
            "Guided drafting. Provide structure, step-by-step guidance, short examples, "
            "and concrete next actions."
        ),
    },
    "Creative": {
        "temp": 1,
        "instr": (
            "Brainstorm innovative topics and angles. Diverge with multiple ideas, then "
            "converge to 2–3 concrete candidates with crisp research questions and "
            "feasibility notes."
        ),
    },
}

With the following function we will use `MODE_INSTR` together with `PERSONA_MAP` when building the system prompt.

In [27]:
def build_system_prompt(
    mode_label: str,
    persona_label: str,
    task_instructions: str | None = None,
) -> tuple[str, float]:
    mode_instr = MODE_INSTR[mode_label]
    persona = PERSONA_MAP[persona_label]

    parts = []
    if task_instructions:
        parts.append(task_instructions.strip())

    parts.append(
        (
            "You are a thesis assistant.\n\n"
            f"Mode: {mode_label} – {mode_instr}\n"
            f"Persona: {persona_label}.\n"
            f"Style: {persona['instr']}"
        )
    )

    system_prompt = "\n\n".join(parts)
    return system_prompt, persona["temp"]



**Using a helper persona with Qwen:**

In [ ]:
mode = "Research question helper"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = qwen_completion(         
    system_prompt=system_prompt,
    user_prompt=feedback_question,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")



### Qwen3 Next 80B A3B – Research question helper – Helper

Your research question is **clear** and **relevant**, but can be refined for **greater feasibility** and **precision**.

### ✅ Strengths:
- **Clear focus**: You identify the tool (AI writing assistants), the population (bachelor students), and the outcomes (quality, originality).
- **Context-specific**: Mentioning “Swiss university” adds useful geographic and institutional grounding.

### 🔧 Areas for Improvement:

#### 1. **Too broad: “Quality” and “originality” are vague**
→ *Problem*: “Quality” could mean grammar, structure, argumentation, depth, etc. “Originality” could mean plagiarism, idea generation, or voice.
→ *Fix*: Define 1–2 measurable indicators.  
**Example**:  
> *"...influence the coherence of arguments and detectable use of AI-generated phrasing in bachelor students’ essays?"*

#### 2. **Feasibility: How will you measure this?**
→ *Problem*: Assessing “originality” requires tools (e.g., Turnitin AI detection), and “quality” needs rubrics or expert scoring — both require resources.
→ *Fix*: Narrow to what you can realistically assess with limited time/resources.  
**Example**:  
> *"...affect the structural coherence and AI-detection scores of bachelor students’ essays at [University Name]?"*

#### 3. **Consider a comparative angle**
→ *Better yet*: Compare essays written with vs. without AI assistance (if you can collect data).  
**Stronger version**:  
> *“How do essays written with the assistance of ChatGPT differ in structural coherence and AI-detection scores from those written without AI tools among bachelor students at [University Name]?”*

### ✅ Recommended Revised Version:
> **“To what extent do bachelor students’ academic essays written with ChatGPT differ in structural coherence and AI-detection scores from those written without AI assistance at [University Name]?”**

### 🚀 Next Steps:
1. **Define your metrics**: What exactly will you count/evaluate? (e.g., essay length, paragraph flow, Turnitin AI score, instructor rubric scores)
2. **Plan data collection**: Can you get 20–30 anonymized essays (with consent)? Can you survey students on AI use?
3. **Check ethics**: Will your university require IRB approval for collecting student work?

You’re on the right track — just narrow and operationalize! Let me know if you want help designing your methodology next.

**Using an instructor persona with Qwen:**

In [13]:
mode = "Research question helper"
persona = "Supervisor"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = qwen_completion(           
    system_prompt=system_prompt,
    user_prompt=feedback_question,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### Qwen3 Next 80B A3B – Research question helper – Supervisor

**Feedback on Research Question:**

**Clarity: ✅ Mostly clear, but vague on key constructs.**  
You name “AI writing assistants” and “quality and originality,” but these are *not operationalized*. What does “quality” mean? Grammar? Argument coherence? Structure? Citations? “Originality” is even more problematic—is it about plagiarism detection scores? uniqueness of phrasing? conceptual novelty? Without definitions, your question is unmeasurable.

**Feasibility: ⚠️ High risk of measurement failure.**  
You’re asking to assess *academic essays* at a *Swiss university*—but you haven’t specified:  
- Which university? (BFH requires institutional access and ethics approval.)  
- How many essays? How will you sample? (Random? Self-selected? Pre/post AI use?)  
- How will you *measure* quality and originality? Rubrics? Turnitin? Expert blind grading? AI detectors?  
Without this, your methodology is a black box. BFH expects *transparent, replicable design*.

**Improvement Suggestions:**

1. **Operationalize variables:**  
   → *“Quality”* = scored using a 5-point rubric on argument structure, clarity, and academic tone (adapted from [cite standard rubric]).  
   → *“Originality”* = measured via Turnitin Similarity Index *and* AI-detection tool (e.g., GPTZero) scores, plus expert judgment on phrasing novelty.

2. **Specify context and method:**  
   → *“...among Bachelor students in Business Administration at BFH during the Spring 2025 semester, comparing essays written with and without AI assistance (n=60), controlled for prior writing proficiency.”*

3. **Narrow scope:**  
   “AI writing assistants” is too broad. Specify: *“ChatGPT (GPT-4) used for drafting or editing”* — not just “use.”

**Revised Example:**  
> *“To what extent does the use of ChatGPT for drafting undergraduate business essays at BFH influence essay quality (as scored by a standardized academic rubric) and originality (as measured by Turnitin and GPTZero) among students with comparable prior writing ability?”*

**BFH Compliance Check:**  
- ✅ Clear population (BFH students)  
- ❌ Missing ethics plan (you must submit to data protection officer if collecting student work)  
- ❌ No mention of control group or pre-test  
- ❌ No justification for tool selection (why GPTZero? Why not ZeroGPT?)

---

**Self-Reflection Checklist (BFH Thesis Standard):**  
☐ Have I defined *all* key constructs with measurable indicators?  
☐ Have I specified the exact population, setting, and time frame?  
☐ Have I identified a feasible data collection method (e.g., essay collection, scoring protocol)?  
☐ Have I addressed ethical/data protection requirements (BFH DPO)?  
☐ Have I ruled out confounding variables (e.g., prior writing skill, course level)?  
☐ Is my question answerable within 3–4 months with bachelor-level resources?

**Next Step:** Draft your *methodology section* in 2 pages. If you can’t describe *how* you’ll measure “quality” and “originality” concretely, your research question is not yet ready. Revise and resubmit.

**Using a creative persona with Qwen:**

In [14]:
mode = "Research question helper"
persona = "Creative"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = qwen_completion(           
    system_prompt=system_prompt,
    user_prompt=feedback_question,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### Qwen3 Next 80B A3B – Research question helper – Creative

Great start! Your research question is **clear**, **relevant**, and **timely** — perfect for a bachelor’s thesis. Here’s a quick breakdown and a few targeted improvements:

---

### ✅ **Strengths:**
- **Clarity:** The variables (AI writing assistants → quality & originality of essays) are well-defined.
- **Feasibility:** You can collect real essays (with consent), use plagiarism/AI-detection tools (e.g., Turnitin, GPTZero), and survey students — all doable in 3–6 months.
- **Context:** Specifying “Swiss university” adds useful geographic/cultural specificity.

---

### 🔧 **Suggestions for Improvement:**

#### 1. **Narrow “quality” and “originality”**  
These are broad. Define them operationally:
- *Quality* = coherence, argument strength, grammar, structure? Use a rubric.
- *Originality* = AI detection scores? Citation accuracy? Novelty of ideas?  
→ *Better:*  
> *“...influence the structural coherence, grammatical accuracy, and AI-detected originality of bachelor students’ academic essays…”*

#### 2. **Add a comparative angle (strongly recommended)**  
To make findings more robust and publishable, compare:
- Essays written *with* vs. *without* AI assistance  
- Or: students who *use* AI vs. those who *don’t*

→ *Stronger version:*  
> *“How does the use of AI writing assistants (e.g., ChatGPT) affect the structural coherence, grammatical accuracy, and AI-detected originality of bachelor students’ academic essays at a Swiss university, compared to essays written without AI assistance?”*

#### 3. **Consider ethical/legal nuance (Swiss context!)**  
Swiss universities have strict data privacy laws (GDPR-like). Mention how you’ll ensure anonymization and consent — this strengthens feasibility and ethics approval.

---

### 🎯 **Final Recommended Version:**
> **“How does the use of AI writing assistants (e.g., ChatGPT) affect the structural coherence, grammatical accuracy, and AI-detected originality of bachelor students’ academic essays at a Swiss university, compared to essays written without AI assistance?”**

### 💡 Bonus: Add one qualitative layer for depth
Ask students: *“How did you use the AI tool, and what did you change or keep?”*  
→ This turns your thesis from “does AI hurt originality?” into “*how* do students negotiate AI in their writing?” — more innovative and insightful.

---

You’re on a very strong track. With these tweaks, your question becomes **precise, measurable, ethically grounded, and analytically rich** — perfect for a standout bachelor thesis. 🚀

<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    It is clearly noticeable that personas have a strong impact on the outputs generated by
    <code>Qwen</code>. The supervisor persona produces more critical responses, while the
    creative persona introduces new ideas. Compared to earlier runs, the answers also tend
    to be longer and more detailed.
  </div>
</div>


**Using a helper persona with GPT:**

In [20]:
mode = "Research question helper"
persona = "Helper"
model_name = "GPT‑OSS 120B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = gpt_oss_completion(         
    system_prompt=system_prompt,
    user_prompt=feedback_question,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### GPT‑OSS 120B – Research question helper – Helper

Below is a quick, structured review of your draft question, followed by concrete suggestions and next‑step actions you can take right away.

---

## 1. Quick Evaluation

| Dimension | What’s strong | What needs work |
|-----------|---------------|-----------------|
| **Clarity** | • The core variables are identified (AI writing assistants, quality, originality, bachelor‑level essays). <br>• The context (Swiss university) is specified. | • “Quality” and “originality” are vague – readers need to know *how* you will measure them. <br>• “Influence” could imply correlation, causation, or perception; the causal direction isn’t explicit. |
| **Feasibility** | • Data can be collected from a single university (e.g., essay submissions, surveys). <br>• AI tools like ChatGPT are readily accessible for experimental manipulation. | • Access to a sufficient number of essays *and* the ability to check originality (e.g., plagiarism detection) may require ethics approval and technical resources. <br>• Controlling for confounding variables (e.g., prior writing skill, subject matter) can be challenging. |
| **Scope** | • Focuses on a specific student group (bachelor students) and a concrete output (academic essays). | • The question tries to cover two outcomes (quality **and** originality) in one go, which may dilute analytical depth. |

---

## 2. How to Strengthen the Question

### A. Pin down the constructs

| Construct | Possible operationalisation |
|-----------|------------------------------|
| **Quality** | • Rubric scores (argumentation, structure, referencing) given by blind graders.<br>• Automated readability metrics (e.g., Coh‑Metrix, Flesch‑Kincaid). |
| **Originality** | • Turn‑itin / iThenticate similarity index.<br>• Human rating of “novelty of argument” on a Likert scale.<br>• Detection of AI‑generated text (e.g., OpenAI’s classifier). |

### B. Clarify the causal direction

- **Descriptive**: “What is the relationship between AI‑assistant use and essay quality/originality?”  
- **Experimental**: “Does prompting students to use ChatGPT improve (or diminish) essay quality/originality compared with a control group?”  

### C. Narrow the scope (optional)

- Focus on **one** outcome first (e.g., quality) and treat originality as a secondary analysis.  
- Or compare **two conditions** (AI‑assisted vs. non‑assisted) rather than trying to capture a continuum of usage.

### D. Refine wording

A more precise version could read:

> **Experimental version**  
> “To what extent does the use of AI writing assistants (e.g., ChatGPT) affect the rubric‑based quality and plagiarism‑detected originality of bachelor‑level academic essays at the University of [Name] in Switzerland?”

> **Descriptive version**  
> “What is the relationship between self‑reported use of AI writing assistants (e.g., ChatGPT) and (a) rubric‑based quality scores and (b) similarity‑index originality measures of bachelor‑level academic essays at a Swiss university?”

Both versions make the *how* (measurement) and *what* (relationship) explicit.

---

## 3. Immediate Next Steps

| Step | What to do | Why it matters |
|------|------------|----------------|
| **1. Define measurement instruments** | • Draft a grading rubric (or adopt an existing one). <br>• Choose a plagiarism/originality detection tool. | Gives you concrete variables for data collection and analysis. |
| **2. Decide on study design** | • **Experimental**: Randomly assign students to “AI‑assisted” vs. “no‑AI” groups. <br>• **Observational**: Survey students about AI use and collect their essays. | Determines the level of causal inference you can claim. |
| **3. Scope the sample** | • Estimate the number of essays needed for statistical power (e.g., using G*Power). <br>• Identify a course or department where you can recruit participants. | Ensures feasibility and ethical approval feasibility. |
| **4. Draft a brief ethics checklist** | • Consent for essay analysis. <br>• Anonymisation plan. <br>• Handling of potentially plagiarised work. | Early ethics clearance saves time later. |
| **5. Write a 2‑sentence working question** | Use the refined wording above, then test it on a peer or supervisor. | A concise version helps you stay focused while you flesh out the full proposal. |

---

## 4. Quick Checklist for Your Final Question

- **[ ]** Specifies *who* (bachelor students at a Swiss university).  
- **[ ]** Specifies *what* (AI writing assistants, e.g., ChatGPT).  
- **[ ]** States *which outcomes* (quality, originality) **and** how they will be measured.  
- **[ ]** Indicates the *type of relationship* (correlation vs. causal effect).  
- **[ ]** Is narrow enough to be answered with the data you can realistically collect.

---

### TL;DR

Your current question is on the right track but needs clearer definitions of “quality” and “originality,” a decision on whether you’re looking at correlation or causation, and possibly a tighter focus on one outcome. Refine the wording as shown above, then move quickly to lock down measurement tools, decide on an experimental vs. observational design, and sketch a sample‑size plan. Once those pieces are in place, the question will be both crystal‑clear and doable. Good luck! 🚀

In [21]:
mode = "Research question helper"
persona = "Supervisor"
model_name = "GPT‑OSS 120B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = gpt_oss_completion(         
    system_prompt=system_prompt,
    user_prompt=feedback_question,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### GPT‑OSS 120B – Research question helper – Supervisor

**Supervisor‑style critique of the proposed research question**

> **“How does the use of AI writing assistants (such as ChatGPT) influence the quality and originality of bachelor students' academic essays at a Swiss university?”**

---

### 1. Clarity  

| Issue | Why it matters | Suggested fix |
|-------|----------------|---------------|
| **Vague constructs** – “quality” and “originality” are not defined. | Without operational definitions the reader cannot know what is being measured or compared. | Specify dimensions (e.g., rubric‑based writing quality: argumentation, structure, language accuracy; originality: plagiarism score, citation density, novelty of ideas). |
| **“AI writing assistants”** – you list ChatGPT but the term could include a range of tools. | Ambiguity leads to inconsistent exposure among participants. | Define the tool(s) explicitly (e.g., “ChatGPT‑4 via the OpenAI web interface”) or treat “AI writing assistants” as a category with clear inclusion criteria. |
| **“Bachelor students”** – no discipline, year, or language specification. | Different faculties (e.g., humanities vs. engineering) have divergent writing conventions. | Limit to a specific faculty or language of instruction (e.g., “German‑language bachelor programmes in the Faculty of Business Administration”). |
| **“Swiss university”** – you do not name the institution. | Institutional policies on AI use differ; anonymity may be required for ethics, but the scope must be clear. | State “at the Bern University of Applied Sciences (BFH)” if that is the host, or keep “a Swiss university” but justify the choice in the methodology. |

---

### 2. Feasibility  

| Concern | Impact on feasibility | Remedy |
|---------|----------------------|--------|
| **Access to essays** – collecting a sufficient sample of essays before and after AI use may be logistically heavy. | Requires cooperation from lecturers, consent from students, and possibly a longitudinal design. | Use a controlled experimental design: recruit a small cohort (n≈30–40) and assign them to AI‑assisted vs. control conditions. |
| **Measuring “influence”** – causal inference is difficult in an observational setting. | Threats to internal validity (self‑selection, prior writing ability). | Implement random assignment or at least matched‑pair design; pre‑test writing ability to control for baseline differences. |
| **Ethical constraints** – BFH’s research ethics board may restrict the use of AI tools that could be considered cheating. | May limit the scope or require a “simulated” AI‑assistance condition. | Obtain explicit ethical approval, frame the study as “educational support” rather than “cheating,” and ensure all participants are fully informed. |
| **Timeframe** – a bachelor thesis typically allows 3–4 months of data collection. | Recruiting, training, and evaluating essays could exceed this window. | Narrow the sample (e.g., one semester, one course) and use existing essay corpora if available. |

---

### 3. Operationalisation & Measurement  

| Variable | Operational definition | Measurement instrument |
|----------|------------------------|------------------------|
| **AI assistance** | Binary (used vs. not used) *or* dosage (number of prompts, word count generated). | Log files from the AI interface; self‑report questionnaire on usage frequency. |
| **Writing quality** | Composite score based on a validated rubric (e.g., Toulmin argumentation, cohesion, grammar). | Blind rating by two independent academic staff; inter‑rater reliability (Cohen’s κ > .80). |
| **Originality** | (a) Plagiarism similarity index (Turnitin/Urkund) < 15 %; (b) Novelty of argument measured by expert rating. | Automated similarity report + expert rubric. |
| **Control variables** – prior GPA, native language, discipline, time spent on essay. | Collected via questionnaire and university records. | Include as covariates in statistical models. |

---

### 4. BFH (Bern University of Applied Sciences) compliance gaps  

| Requirement | Potential gap | How to address |
|-------------|---------------|----------------|
| **Ethics approval** – BFH’s Ethics Committee requires a detailed risk assessment for AI‑assisted writing. | No mention of consent, data protection, or risk of academic misconduct. | Draft a consent form, anonymise essays, and include a mitigation plan (e.g., de‑briefing, plagiarism checks). |
| **Data protection (DSGVO)** – student essays are personal data. | No data‑handling plan. | Store data on BFH‑approved servers, encrypt files, limit access to the research team. |
| **Academic integrity policy** – BFH has explicit rules on AI usage in assessments. | Study may conflict with institutional policy. | Align the experimental condition with a “learning‑support” scenario approved by the faculty, or conduct the study in a non‑graded, simulated assignment. |
| **Methodological rigor** – BFH expects a clear, replicable methodology. | Current question lacks methodological detail. | Include a concise methods section in the proposal (design, sample size calculation, analysis plan). |

---

### 5. Revised research question (example)

> **“What is the effect of controlled use of ChatGPT‑4 on rubric‑based writing quality and plagiarism‑detected originality in German‑language bachelor essays within the Faculty of Business Administration at BFH?”**

*This version:*
- Specifies the AI tool and usage condition.  
- Defines the outcome measures.  
- Delimits the population and institutional context.  
- Facilitates a causal (experimental) design.

---

## Next steps (concrete)

1. **Literature scan (2 weeks)** – Identify existing validated writing‑quality rubrics and studies on AI‑assisted writing in higher education.  
2. **Operational definition draft (1 week)** – Write precise definitions for “quality” and “originality” and select measurement tools.  
3. **Pilot study (2 weeks)** – Recruit 5–6 students, have them write an essay with/without ChatGPT, collect data, test inter‑rater reliability.  
4. **Ethics application (parallel to pilot)** – Prepare consent forms, data‑management plan, and risk assessment for BFH’s Ethics Committee.  
5. **Power analysis (1 week)** – Estimate required sample size (e.g., using G*Power, α = .05, power = .80, medium effect size d ≈ 0.5 → n ≈ 64).  
6. **Full data collection (4–6 weeks)** – Implement random assignment, log AI usage, collect essays, run plagiarism checks.  
7. **Statistical analysis plan (1 week)** – Pre‑register mixed‑effects model controlling for covariates.  
8. **Write up** – Follow BFH thesis guidelines (structure, citation style, appendices).  

---

## Self‑reflection checklist (for you)

- [ ] **Construct clarity:** Have I defined *quality* and *originality* in measurable terms?  
- [ ] **Scope limitation:** Does the question focus on a single faculty, language, and AI tool?  
- [ ] **Feasibility:** Can I recruit the required sample within the thesis timeline?  
- [ ] **Ethics & data protection:** Are consent, anonymity, and compliance with BFH policies addressed?  
- [ ] **Methodological rigor:** Is the design (experimental vs. observational) capable of answering the causal “how does … influence” query?  
- [ ] **Statistical power:** Have I performed a power analysis to justify the sample size?  
- [ ] **Replication potential:** Could another researcher repeat the study using my protocol?  

If any box remains unchecked, revisit the corresponding element before finalising your proposal. Good luck!

In [22]:
mode = "Research question helper"
persona = "Creative"
model_name = "GPT‑OSS 120B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = gpt_oss_completion(         
    system_prompt=system_prompt,
    user_prompt=feedback_question,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### GPT‑OSS 120B – Research question helper – Creative

**Quick feedback on your draft question**

| Dimension | What’s working | What could be tightened |
|-----------|----------------|--------------------------|
| **Clarity** | ✔︎ The “who” (bachelor students), “what” (AI writing assistants) and “where” (a Swiss university) are all spelled out. | – “Quality” is a broad umbrella. Do you mean argumentation, grammar, coherence, citations, or something else?<br>– “Originality” could refer to plagiarism avoidance, novelty of ideas, or stylistic distinctiveness. |
| **Feasibility** | ✔︎ You can recruit a single cohort from one university, run a controlled experiment or a quasi‑experimental design, and collect essays for analysis. | – Measuring “quality” and “originality” reliably needs validated rubrics or automated tools; establishing inter‑rater reliability will take time.<br>– If you want to compare *before* and *after* AI‑assistance, you’ll need a baseline sample that didn’t use the tool—possible, but you must control for prior writing skill. |
| **Improvement tips** | – Specify *how* you’ll operationalise the constructs.<br>– Clarify whether you’re interested in *perceived* quality (students’ self‑assessment) or *objective* quality (rater scores).<br>– Consider narrowing the scope to a single discipline (e.g., social sciences) or a specific type of essay (e.g., literature review) to keep the project manageable. |

---

## How to sharpen the question

1. **Pick an outcome metric**  
   *Example*: “lexical sophistication”, “argument structure rating”, “Turnitin similarity score”, or “human rater originality score”.

2. **Define the AI‑assistance condition**  
   - “Using ChatGPT to generate a full first draft” vs. “using ChatGPT only for phrasing suggestions”.  
   - Provide a brief description of the prompting protocol (e.g., “students asked ChatGPT to rewrite each paragraph in their own words”).

3. **Set the comparison**  
   - Experimental (AI‑assisted vs. non‑assisted group).  
   - Within‑subject (same students write two essays, one with AI, one without, in counter‑balanced order).

4. **Make the scope explicit**  
   - One faculty (e.g., Business Administration) at *University X* in *Fall 2025*.  
   - A single assignment type (e.g., 1500‑word argumentative essay).

### Revised concrete candidates

| # | Revised research question | Why it’s tighter |
|---|---------------------------|------------------|
| **A** | *“To what extent does using ChatGPT for drafting improve (a) linguistic quality and (b) originality, as measured by expert rubric scores, in 1500‑word argumentative essays written by Bachelor‑level Business Administration students at the University of Zurich?”* | • Breaks “quality” into a measurable dimension (linguistic quality).<br>• Adds a concrete essay type and discipline.<br>• States the assessment method (expert rubric). |
| **B** | *“How does prompting ChatGPT for sentence‑level paraphrasing affect the Turnitin similarity index and the perceived originality of Bachelor theses in the Faculty of Humanities at ETH Zurich?”* | • Focuses solely on originality via a widely used plagiarism detection metric.<br>• Uses a clear, easily‑collected outcome (similarity index). |
| **C** | *“What differences in argument structure and citation accuracy are observed between essays written with and without AI writing assistants among first‑year psychology majors at a Swiss university?”* | • Targets two specific quality aspects that can be reliably coded.<br>• Limits the population to first‑year students, reducing variance from prior coursework. |

---

## Next steps you can take right now

1. **Choose your primary outcome(s).**  
   – List all possible quality/originality indicators (rubric scores, plagiarism scores, readability indices, peer‑rating).  
   – Pick 1‑2 that you can realistically obtain within the semester.

2. **Sketch a pilot design.**  
   – Recruit 20–30 students (or use a class you already teach).  
   – Randomly assign them to “AI‑assisted” vs. “control” condition.  
   – Collect essays, have two blind raters apply your rubric, compute inter‑rater reliability (Cohen’s κ > 0.7).

3. **Check data‑availability & ethics.**  
   – Does your university’s ethics board have a fast‑track for studies involving student writing?  
   – Are there existing rubric templates (e.g., from the university’s writing center) you can adapt?

4. **Write a short “operationalisation” paragraph** for your proposal:  
   *e.g., “Quality will be operationalised as the sum of scores on the University‑approved Writing Assessment Rubric (coherence, evidence use, grammar). Originality will be operationalised as the Turnitin similarity percentage after removing quotations and references.”*

5. **Iterate the wording** based on the feedback you get from your supervisor or peers—keep the final question under 30 words, with a clear subject, intervention, comparator, and outcome (the classic **SICO** structure).

---

### Quick checklist before you lock it in

- [ ] **Specific** – Who, what, where, when?  
- [ ] **Measurable** – Do you have a reliable instrument?  
- [ ] **Attainable** – Can you collect enough essays in the semester?  
- [ ] **Relevant** – Does the question matter to teachers, students, or policy?  
- [ ] **Time‑bounded** – Is the scope limited to a single course/semester?

If you can tick all five, you’re ready to move from a *nice‑to‑explore* idea to a solid, doable bachelor‑level research project. Good luck, and let me know which direction you decide to follow—we can flesh out the methodology or brainstorm the rubric together!

<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    Personas appear to have a slightly weaker impact on the outputs generated by
    <code>GPT</code>. Critical feedback is formulated more indirectly and often presented
    as suggestions. Overall, the responses contain many recommendations and checklist-style
    elements. The answers are consistently long and frequently include tables, largely
    independent of the selected persona.
  </div>
</div>


## RESEARCH QUETION ASSISTANT

Now lets focus on the agents functionalities and start with the research question assitance. Papers need to be summarised, questions must be answered based one retrieved docks and research gaps need to be identified.
In this section we will show you how the prompts are improved and how our 2 LLM's perfomance has changed.

In [23]:
paper_description = """
Below is a short abstract of a research paper.

"AI writing assistants are increasingly used by undergraduate students.
This study investigates how access to an AI writing assistant (similar to
ChatGPT) affects the quality and originality of short academic essays.
In a quasi-experimental design, one group of students could use the AI
assistant while writing, while a comparison group wrote without AI.
Essays were scored with an analytic rubric and checked with plagiarism
software. Survey data captured students' perceived usefulness and concerns."

Please help me understand what this paper does.
"""

 ### 1) PAPER_QUESTION WITHOUT PROMPT ENGINEERING

In [24]:
mode = "Research question helper"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = qwen_completion(         
    system_prompt=system_prompt,
    user_prompt=paper_description,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### Qwen3 Next 80B A3B – Research question helper – Helper

Absolutely — let’s break this down step by step so you fully understand what this paper does.

---

### ✅ **What This Paper Does (In Simple Terms):**

This study **tests whether using an AI writing assistant (like ChatGPT) changes the quality and originality of undergraduate students’ short academic essays.**

It’s not just asking *if* students use AI — it’s asking:  
> **“When students use AI to help write essays, do their essays get better? Or do they become less original (i.e., more plagiarized or less their own work)?”**

---

### 🔍 **Key Components Explained:**

| Element | What It Means |
|--------|---------------|
| **Participants** | Undergraduate students — likely from one or more university courses. |
| **Design** | **Quasi-experimental** → Not fully randomized (e.g., students weren’t randomly assigned to groups, maybe by class or section), but still has a comparison group. |
| **Two Groups** | - **AI Group**: Could use an AI writing assistant while writing.<br>- **Control Group**: Wrote without any AI help. |
| **What Was Measured** | 1. **Quality** → Scored using an *analytic rubric* (e.g., clarity, structure, argument, grammar).<br>2. **Originality** → Checked with *plagiarism software* (like Turnitin) to detect AI-generated or copied content. |
| **Additional Data** | Surveys asked students: *“Did you find the AI helpful?” “Did you worry about ethics or cheating?”* → Gives context to the numbers. |

---

### 🎯 **The Core Research Question (What They’re Trying to Answer):**

> **Does access to an AI writing assistant improve the quality of students’ academic essays — and/or does it reduce their originality?**

This is a *dual-effect* question:  
- Could AI be a **tool for learning** (better quality)?  
- Or a **risk for academic dishonesty** (less originality)?

---

### 💡 Why This Matters:

- AI writing tools are exploding in use — but we don’t yet know their real impact on learning.
- Universities are scrambling to create policies. This study gives **evidence-based insights**.
- It doesn’t just say “AI is good/bad” — it measures **actual outcomes** (essay scores + plagiarism) + student perceptions.

---

### ✅ Next Step for You (If You’re Building on This):

If you’re thinking of doing your own research based on this, here’s a **refined research question** you could use or adapt:

> **“How does access to AI writing assistants affect the quality and originality of undergraduate academic essays, and how do students’ perceptions of AI use correlate with these outcomes?”**

Or, if you want to narrow it further:

> **“Among first-year undergraduates, does using ChatGPT during essay writing lead to higher-quality essays as rated by instructors, and does it increase the likelihood of AI-generated content flagged by plagiarism detectors?”**

---

### 🚀 Your Next Action:

👉 **Pick one of these two:**

1. **If you’re writing a paper**: Use the refined research question above as your guiding question.
2. **If you’re designing a study**: Think — *How could you improve this design?*  
   - Random assignment?  
   - Longer essays?  
   - Multiple AI tools?  
   - Tracking revision history?

Let me know which direction you’d like to go — I’ll help you build it out!

### 2) PAPER_QUESTION WITH PROMPT ENGINEERING

In [52]:
engineered_system = """
You are a thesis assistant helping a student understand a research paper.

When the user gives you an abstract or short description of a paper, ALWAYS structure
your answer with the following headings:

1. Topic / problem
2. Research question(s) (if visible or implied)
3. Methodology (design, participants, measures)
4. Data
5. Key findings (only what is clearly supported)
6. Limitations / gaps
7. How this paper could be useful for a bachelor thesis

Important rules:
- Do NOT invent details that are not clearly supported by the abstract.
- If something is not stated, say "not specified in the abstract".
- Write clearly and concisely, so the student can reuse parts in their thesis notes.
"""

mode = "Research question helper"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona, engineered_system)

answer = qwen_completion(         
    system_prompt=system_prompt,
    user_prompt=paper_description,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### Qwen3 Next 80B A3B – Research question helper – Helper

Absolutely! Let’s break this down step by step using your requested structure — this will help you understand the paper clearly and use it as a foundation for your bachelor thesis.

---

**1. Topic / problem**  
The topic is the impact of AI writing assistants (like ChatGPT) on undergraduate students’ academic writing.  
The problem: As AI tools become widely available, there is growing concern about whether they improve writing quality or undermine originality and academic integrity.

---

**2. Research question(s)**  
The paper implies two clear research questions:  
- *How does access to an AI writing assistant affect the **quality** of undergraduate students’ short academic essays?*  
- *How does access to an AI writing assistant affect the **originality** (i.e., risk of plagiarism or unoriginal content) of those essays?*  

There’s also an exploratory question implied by the survey data:  
- *What do students perceive as the usefulness and risks of using AI writing assistants?*

---

**3. Methodology (design, participants, measures)**  
- **Design**: Quasi-experimental (not randomized — likely students chose or were assigned to AI or no-AI groups based on class or availability).  
- **Participants**: Undergraduate students (exact number, field, or institution not specified).  
- **Measures**:  
  - *Outcome 1*: Essay quality → scored using an **analytic rubric** (likely including criteria like clarity, structure, argument, grammar).  
  - *Outcome 2*: Originality → checked with **plagiarism detection software** (e.g., Turnitin).  
  - *Supplementary data*: **Survey responses** on perceived usefulness and concerns (Likert-scale or open-ended questions).

---

**4. Data**  
- Quantitative: Essay scores from the rubric + plagiarism detection results (e.g., similarity percentages).  
- Qualitative: Open-ended or scaled survey responses about student attitudes toward AI use.

---

**5. Key findings (only what is clearly supported)**  
*The abstract doesn’t state the actual results*, so we can’t list findings — only that the study *measured* them.  
But we can infer what they likely looked for:  
→ Did the AI group write higher-quality essays?  
→ Did they have higher plagiarism scores?  
→ Did students report finding AI helpful but also worrying about over-reliance?

*Note: You’d need to read the full paper to know the actual outcomes.*

---

**6. Limitations / gaps**  
From the abstract, we can identify:  
- **No random assignment** → selection bias possible (e.g., more motivated students may have chosen to use AI).  
- **Short essays only** → findings may not generalize to longer research papers.  
- **One AI tool** → results may not apply to all AI writing assistants.  
- **Self-reported perceptions** → surveys may reflect bias or social desirability.  
- **No long-term effects** → didn’t measure learning outcomes over time.

---

**7. How this paper could be useful for a bachelor thesis**  
This paper is **excellent** for a bachelor thesis because:  
✅ It’s **focused and feasible** — you can replicate or extend it with a small sample.  
✅ It uses **clear, measurable outcomes** (rubric scores + plagiarism software) — easy to implement.  
✅ It combines **quantitative + qualitative data** — great for mixed-methods theses.  
✅ It addresses a **timely, real-world issue** — AI in education is hot right now.  

**Concrete next steps for your thesis**:  
1. **Refine your research question**:  
   → *“To what extent does access to ChatGPT improve the quality and reduce originality of undergraduate essays in [your field]?”*  
2. **Adapt the methodology**:  
   → Recruit 30–50 classmates.  
   → Assign half to write with AI, half without.  
   → Use a simple 5-point rubric (you can adapt one from the paper).  
   → Run essays through free plagiarism checker (e.g., QuillBot, Grammarly, or Turnitin if available).  
   → Add a 5-question survey: “Did AI help you structure your essay?” “Did you feel it reduced your learning?” etc.  
3. **Compare your results** to this paper — did you find similar patterns? Why or why not?  

---

**Want help drafting your own research question?**  
Here’s a template you can fill in:  
> *“How does [use of AI tool] affect [writing quality / originality / student confidence] among [your target group] when writing [type of assignment] in [your discipline]?”*

Let me know your field (e.g., psychology, business, literature) and I’ll help you tailor it!

<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    The paper summary now follows a clear and consistent structure. This structure helps
    students focus on key aspects such as the research problem, methodology, and limitations.
    By keeping this format consistent, different studies become easier to compare and
    relevant similarities and differences can be identified more clearly.
  </div>
</div>


Lets also have a look at GPTs answer:

In [53]:
mode = "Research question helper"
persona = "Helper"
model_name = "GPT‑OSS 120B"

system_prompt, temp = build_system_prompt(mode, persona, engineered_system)

answer = gpt_oss_completion(         
    system_prompt=system_prompt,
    user_prompt=paper_description,
    temperature=0.2,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### GPT‑OSS 120B – Research question helper – Helper

**1. Topic / Problem**  
The paper examines the impact of AI writing assistants (e.g., tools similar to ChatGPT) on undergraduate students’ academic writing, specifically looking at *essay quality* and *originality*.

**2. Research Question(s)**  
- How does access to an AI writing assistant affect the quality of short academic essays written by undergraduate students?  
- How does access to an AI writing assistant affect the originality (plagiarism risk) of those essays?  
(These questions are implied from the abstract; exact wording is not provided.)

**3. Methodology (design, participants, measures)**  
- **Design:** Quasi‑experimental (one treatment group with AI assistance, one control group without).  
- **Participants:** Undergraduate students (sample size not specified).  
- **Measures:**  
  - Essay quality: scored using an analytic rubric.  
  - Originality: assessed with plagiarism‑detection software.  
  - Student perceptions: collected via a survey on usefulness and concerns about the AI tool.

**4. Data**  
- Written essays from both groups (content for rubric scoring and plagiarism checking).  
- Survey responses on perceived usefulness and concerns.  
- No details on sample size, demographic characteristics, or statistical methods are given in the abstract.

**5. Key Findings**  
*Not specified in the abstract.* (The abstract only describes what was investigated, not the results.)

**6. Limitations / Gaps**  
- Quasi‑experimental design may limit causal inference (e.g., non‑random assignment).  
- No information on participant demographics, sample size, or how groups were formed.  
- The abstract does not mention long‑term effects, different essay topics, or variations in AI tool usage (e.g., frequency, depth of interaction).  
- Originality assessment relies solely on plagiarism software, which may not capture nuanced issues like “AI‑generated but uncited ideas.”

**7. How This Paper Could Be Useful for a Bachelor Thesis**  
- **Literature foundation:** Provides a recent empirical study on AI writing assistants in higher education, useful for framing the background and justification of your own work.  
- **Methodological template:** Shows a feasible quasi‑experimental setup, rubric‑based quality assessment, and plagiarism detection—methods you could adapt or improve.  
- **Research gaps to explore:**  
  - Randomized controlled designs to strengthen causal claims.  
  - Longitudinal effects of AI assistance on writing skill development.  
  - Differences across disciplines, essay lengths, or student proficiency levels.  
  - Qualitative analysis of how students interact with the AI (e.g., prompts used).  
- **Survey instrument:** The perceived usefulness/concern questionnaire could be a starting point for your own attitude measures.

---

## Next Steps for Defining Your Own Research Question

1. **Clarify Your Interest Area**  
   - Are you more interested in *learning outcomes* (e.g., skill development), *ethical concerns* (e.g., plagiarism), or *pedagogical practices* (e.g., how instructors can integrate AI tools)?

2. **Identify a Feasible Scope**  
   - **Population:** One course, one discipline, or a specific year group?  
   - **Intervention:** Full access to ChatGPT, limited prompts, or a specific feature (e.g., outline generation).  
   - **Outcome Measures:** Quality rubric, originality score, self‑efficacy, time on task, etc.

3. **Draft a Precise Research Question**  
   Example formats:  
   - *“What is the effect of limited‑prompt access to ChatGPT on the analytical writing quality of first‑year psychology students?”*  
   - *“How do undergraduate engineering students perceive the usefulness of AI writing assistants for literature reviews, and how does this perception relate to plagiarism scores?”*

4. **Choose an Appropriate Design**  
   - **Randomized Controlled Trial (RCT):** Randomly assign students to AI vs. no‑AI groups for stronger causal inference.  
   - **Mixed‑Methods:** Combine quantitative rubric scores with qualitative interviews about AI usage strategies.  

5. **Plan Data Collection**  
   - Determine sample size (power analysis).  
   - Select or develop a validated writing rubric.  
   - Choose plagiarism detection software and set similarity thresholds.  
   - Design a survey (adapt items from the paper’s questionnaire, adding any missing constructs you care about).

6. **Consider Ethical Issues**  
   - Obtain informed consent, especially regarding AI usage and plagiarism detection.  
   - Ensure transparency about how AI assistance is logged (e.g., screen recordings, prompt logs) while respecting privacy.

7. **Outline Your Timeline**  
   - **Weeks 1‑2:** Literature review (including this paper).  
   - **Weeks 3‑4:** Finalize research question, design, and instruments.  
   - **Weeks 5‑6:** Ethics approval and pilot testing.  
   - **Weeks 7‑10:** Data collection (essay writing sessions, surveys).  
   - **Weeks 11‑13:** Scoring, plagiarism checks, data analysis.  
   - **Weeks 14‑16:** Write up results and discussion.

---

### Quick Check‑list for Your Proposal

- [ ] Clear, focused research question (e.g., effect of AI on *X*).  
- [ ] Justified need (cite this paper as evidence of a gap).  
- [ ] Feasible sample and setting (e.g., one class, 30–40 students).  
- [ ] Defined outcome measures (rubric, plagiarism score, survey).  
- [ ] Appropriate design (RCT, quasi‑experiment, mixed‑methods).  
- [ ] Ethical considerations addressed.  

Feel free to share more about the discipline or specific angle you want to pursue, and I can help you refine the question further or suggest concrete measurement tools!

<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    The results are very similar to those obtained with <code>Qwen</code>. Both models are suitable for this task.
  </div>
</div>


 ### Answering general guidance

For the next cells we will compare general guidance with and without RAG with the 2 LLMs and will show you our prompts for RAG functionalities

In [63]:
general_q = """
I want to study the impact of AI writing assistants on the quality
of bachelor students' academic essays. How should I proceed?
"""

# CONDITION A: NO BFH / CRESWELL (NO RAG)
mode = "Research question helper"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = qwen_completion(         
    system_prompt=system_prompt,
    user_prompt=general_q,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### Qwen3 Next 80B A3B – Research question helper – Helper

Great start! You’re tackling a timely and relevant topic. To turn your broad interest into a **feasible, precise, and researchable question**, let’s walk through this step by step.

---

### ✅ STEP 1: Narrow Your Focus — Refine the Research Question

Your current idea:  
> *“The impact of AI writing assistants on the quality of bachelor students’ academic essays.”*

This is too broad. Let’s make it **specific** by answering:

#### 🔍 Clarifying Questions:
1. **Which AI writing assistants?**  
   – e.g., Grammarly, ChatGPT, Copyscape, QuillBot, or a general category?  
   → *Choose 1–2 dominant tools used by students in your context.*

2. **What do you mean by “quality”?**  
   – Is it grammar, coherence, originality, argument strength, citation accuracy, depth of analysis?  
   → *Pick 2–3 measurable dimensions.*

3. **Compared to what?**  
   – Essays written without AI? With human tutoring? Before vs. after AI use?  
   → *You need a comparison group or baseline.*

4. **In which discipline?**  
   – Essays in psychology? Literature? Engineering?  
   → *Discipline affects writing standards and AI’s impact.*

5. **When?**  
   – Are you studying essays written during a semester? After training?  
   → *Timeframe matters for data collection.*

---

### ✍️ EXAMPLE REFINED RESEARCH QUESTION:

> *“To what extent does the use of ChatGPT influence the argumentative coherence and originality of bachelor-level essays in English Literature, compared to essays written without AI assistance?”*

✅ This version is:  
- Specific (tool: ChatGPT)  
- Measurable (coherence, originality)  
- Contextual (English Literature, bachelor students)  
- Comparative (with vs. without AI)

---

### ✅ STEP 2: Define Your Methodology — How Will You Measure This?

You’ll need **data**. Here are two feasible approaches:

#### 📊 Option A: Experimental Design (Strongest for causality)
- Recruit 40–60 bachelor students.
- Randomly assign them to:
  - **Group A**: Write essay *without* AI
  - **Group B**: Write essay *with* ChatGPT (allowed to use it)
- Give both groups the same prompt (e.g., “Discuss the role of alienation in Kafka’s *The Metamorphosis*”)
- Blindly assess essays using a **rubric** measuring:
  - Argument clarity (1–5 scale)
  - Logical flow/coherence (1–5)
  - Originality (via Turnitin or manual review)
  - Grammar/mechanics (Grammarly-style scoring)

→ *Advantage:* You can claim “AI use caused X change.”

#### 📈 Option B: Observational Study (Easier to implement)
- Collect 30–50 real essays written by students in your university.
- Survey students: “Did you use AI? Which tool? How much?”
- Have 2–3 trained raters assess essay quality using the same rubric.
- Statistically compare quality scores between AI-users and non-users.

→ *Advantage:* More realistic; reflects actual behavior.  
→ *Limitation:* Can’t prove causation (maybe high-ability students use AI more).

> 💡 **Recommendation**: Start with Option B if you’re short on time or access. Use Option A if you can design a controlled study.

---

### ✅ STEP 3: Ethical & Practical Considerations

- **Ethics**: Inform participants. Get IRB/ethics approval if required.
- **Transparency**: Ask students to disclose AI use honestly (anonymize data).
- **Bias alert**: Don’t assume AI = bad. Some students use it to *improve* drafts.
- **Tools**: Use free rubrics like the **AAC&U VALUE Rubrics** (for critical thinking, written communication).

---

### ✅ STEP 4: Next Actions — Your 1-Week Plan

| Day | Task |
|-----|------|
| 1 | Pick **1 AI tool** (e.g., ChatGPT) and **1 academic discipline** (e.g., Sociology) |
| 2 | Define **3 measurable quality dimensions** (e.g., clarity, originality, citation accuracy) |
| 3 | Draft your final research question (use the example above as template) |
| 4 | Choose your method: Experimental or Observational? Justify your choice |
| 5 | Design your essay prompt and rubric (use AAC&U or create your own) |
| 6 | Contact your supervisor or ethics committee — ask: “Is this feasible?” |
| 7 | Start recruiting participants or collecting existing essays |

---

### 📌 BONUS: Suggested Reading

- **“AI Writing Tools and Academic Integrity: A Survey of Student Use”** (Computers & Education, 2023)  
- **“The Impact of AI Writing Assistants on Student Writing Quality”** (Journal of Educational Computing Research, 2024)  
- AAC&U VALUE Rubrics: https://www.aacu.org/value/rubrics

---

### 🚀 Your Turn:
> **Reply with your refined research question and chosen method**, and I’ll help you polish it further or design your rubric!  
> Example:  
> *“I want to study how Grammarly affects grammar and structure in business students’ essays, using an observational design with 40 existing essays.”*

Let’s make this real. I’m here to help you build it.

In [64]:
import importlib, rag_tools
importlib.reload(rag_tools)

from rag_tools import retrieve_kb_context

# CONDITION B: WITH BFH / CRESWELL + PROMPT ENGINEERING
RAG_SAFETY_PREAMBLE = """You are an assistant in a Retrieval-Augmented Generation (RAG) app.

You MUST:
- Use ONLY the information that appears in the [Retrieved Context] section.
- NOT invent authors, titles, dates, numbers of studies, sample sizes, or detailed findings that are not clearly stated.
- If the retrieved text is incomplete for the question, say what is missing and suggest what the student should check in the original documents.
"""

# Retrieve Creswell/BFH guidance for methods / structure_question
flavored_query = general_q + (
    " (research design, validity, reliability, sampling, data collection, "
    "Creswell designs, BFH bachelor thesis requirements)" # In the agent workflow these tags are chosen by the llm in another api call before this step based on the users message
)
docs, metas = retrieve_kb_context(flavored_query, n_results=8, min_bfh=2)
context = "\n\n".join(docs)

engineered_system = (
    "You are a BFH thesis methods assistant in a RAG app. "
    "Follow the instructions and retrieved context in the user message carefully, "
    "and answer in clear, structured markdown."
)

engineered_user = f"""{RAG_SAFETY_PREAMBLE}

[Retrieved Context from Creswell/BFH]
{context}

[Student question]
{general_q}

TASK:
- DO NOT suggest any research questions without having access to relevant papers.
- Return a clear, structured answer (with short headings) that the student can directly use to improve their methods section.
- Be explicit about trade-offs (e.g., internal vs external validity, feasibility vs ambition).
- If the user’s idea is unrealistic for a thesis, say so gently and suggest a more feasible variant.


"""


mode = "Research question helper"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona, engineered_system)

answer = qwen_completion(         
    system_prompt=system_prompt,
    user_prompt=engineered_user,
    temperature=0.1,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Qwen3 Next 80B A3B – Research question helper – Helper

**Step-by-Step Guidance: Studying AI Writing Assistants and Bachelor Students’ Essay Quality**

---

### **1. Clarify Your Research Goal (Feasibility Check)**  
Your goal — *“studying the impact of AI writing assistants on the quality of bachelor students’ academic essays”* — is **ambitious but feasible** if narrowed.  

⚠️ **Trade-off**:  
- *High ambition*: Measuring “impact on quality” requires defining, measuring, and comparing essay quality before/after AI use — this demands controlled conditions, pre/post tests, and expert raters.  
- *High feasibility*: You can study *how students use* AI tools and *perceive* their impact — this is more manageable for a bachelor’s thesis.  

✅ **Recommended Feasible Variant**:  
> *“How do bachelor’s students in [your program] use AI writing assistants (e.g., ChatGPT) when writing academic essays, and how do they perceive the impact on their writing quality?”*  

This shifts focus from *objective quality measurement* (hard to validate without a control group and rubric-based scoring) to *usage patterns and self-perceived effects* — which aligns with qualitative or mixed methods and is realistic within thesis constraints.

---

### **2. Design Your Methodology (Based on BFH Guidelines)**  
Use a **mixed methods approach** — it fits your interest in both behavior (quantitative) and perception (qualitative).  

#### **Quantitative Component (Optional but helpful)**  
- Survey: Ask students (N = 30–50) to rate:  
  - Frequency of AI use (never, rarely, often, always)  
  - Perceived improvement in clarity, structure, grammar, originality (5-point Likert scale)  
  - Confidence in essay quality before/after AI use  

#### **Qualitative Component (Core)**  
- Semi-structured interviews (n = 8–12 students) with:  
  - AI users (at least 2–3 frequent users)  
  - Non-users (for contrast)  
- Prompts:  
  - *“Walk me through how you used ChatGPT/Elicit in your last essay.”*  
  - *“Did you change anything the AI generated? Why or why not?”*  
  - *“Did you worry about academic integrity? How did you handle citations?”*  

📌 **BFH Requirement**: You **must document** all AI tool usage in your thesis (see context).  
> Example: *“This analysis was supported by ChatGPT (OpenAI) for brainstorming essay structures and paraphrasing sentences. All AI-generated text was manually reviewed and rewritten by the author.”*

---

### **3. Address AI Hallucinations & Academic Integrity (Critical!)**  
- **Do NOT cite AI as a source** (BFH: “ChatGPT cannot be cited as an author”).  
- **Mark all AI-generated text** as “personal communication” (BFH example):  
  > *“The introduction was drafted using ChatGPT (OpenAI, personal communication, April 2025) and subsequently revised for academic tone and originality.”*  
- **Verify every reference** AI provides. BFH warns: *“These tools are prone to hallucination… A fact should be considered reliable only when corroborated by an additional source.”*  
- **Use AI only for drafting, brainstorming, or editing** — never for generating literature reviews or claims.

---

### **4. Ensure Validity & Trustworthiness (Qualitative Rigor)**  
Since you’ll rely on student self-reports:  
- **Triangulate**: Compare survey data with actual essay drafts (if you can collect them anonymously).  
- **Peer debriefing**: Ask a fellow student or supervisor to review your interview analysis.  
- **Researcher reflexivity**: In your methods section, write:  
  > *“As a student who has used AI tools myself, I acknowledge potential bias in interpreting participants’ experiences. I kept a reflexive journal to note assumptions during data analysis.”*

---

### **5. What to Avoid (Common Pitfalls)**  
❌ Trying to measure “objective essay quality” with a rubric without trained raters → too resource-heavy.  
❌ Claiming AI “improves” or “hurts” quality without control group → causal claims require experimental design (not feasible for bachelor’s thesis).  
❌ Citing ChatGPT as an author or source → violates BFH academic integrity rules.

---

### **6. Next Steps (Action Plan)**  
1. **Refine your research question** to focus on *usage and perception*, not causal impact.  
2. **Draft a 5-question survey** (use Google Forms).  
3. **Recruit 10–15 participants** from your program (ask classmates, use course mailing lists).  
4. **Prepare 3–5 interview questions** based on the prompts above.  
5. **Write your AI disclosure statement** now — place it in your thesis appendix.  
6. **Get supervisor approval** on your method before data collection.

---

### ✅ Final Note  
You’re not studying whether AI *makes essays better* — you’re studying how students *experience* and *navigate* AI in their academic writing. That’s a meaningful, ethical, and achievable thesis topic under BFH guidelines.  

**You’ve got this.** Start small. Document everything. Stay critical.

<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    Despite the prompt instruction stating that no research questions should be suggested
    without access to relevant papers, the model still proposed a research question. This
    indicates that such constraints are not always followed strictly and need to be enforced
    additionally through system logic.
  </div>
</div>


In [65]:
mode = "Research question helper"
persona = "Helper"
model_name = "GPT‑OSS 120B"

system_prompt, temp = build_system_prompt(mode, persona, engineered_system)

answer = gpt_oss_completion(         
    system_prompt=system_prompt,
    user_prompt=engineered_user,
    temperature=0.1,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### GPT‑OSS 120B – Research question helper – Helper

## 1. Clarify the Scope of Your Study  

| Aspect | What the BFH policy & Creswell’s guidance tell you | Practical implication for your thesis |
|--------|---------------------------------------------------|----------------------------------------|
| **Research focus** | “Impact of AI writing assistants on the quality of bachelor students’ academic essays.” | You need a **clear dependent variable** (essay quality) and an **independent variable** (use of an AI writing assistant). |
| **Audience** | “Researchers write for audiences that will accept their research … journal editors, graduate committees, … colleagues.” | Choose a design that matches the expectations of your supervisors and the Business School’s thesis committee (usually quantitative or mixed‑methods). |
| **Feasibility** | “Collect and analyze both quantitative and qualitative data … fits a person who enjoys both the structure of quantitative research and the flexibility of qualitative inquiry.” | A **mixed‑methods** design can give you richer insight while staying realistic for a bachelor‑level thesis. |

---

## 2. Select an Appropriate Research Design  

### 2.1 Quantitative‑only option  
*Strengths* – High internal validity, easier statistical inference, clear “impact” estimate.  
*Limitations* – May overlook students’ experiences, motivations, and the nuanced ways they use AI tools.  

### 2.2 Qualitative‑only option  
*Strengths* – Deep understanding of how students perceive AI assistance, rich contextual data.  
*Limitations* – Harder to claim “impact” in a causal sense; limited generalisability.  

### 2.3 Mixed‑methods (recommended)  
*Combine* a **controlled experiment** (quantitative) with **focus‑group interviews** (qualitative).  
*Trade‑off* – More work, but you gain both **internal validity** (through the experiment) and **external validity/transferability** (through thick description of student experiences).  

> **Creswell’s guidance**: “Decisions about choice of a design are further influenced by the research problem … and the audience …” – a mixed design satisfies both the need for measurable impact and the audience’s expectation for depth.

---

## 3. Quantitative Component – Measuring “Impact”

| Step | Guidance from the retrieved text | How to apply it |
|------|----------------------------------|-----------------|
| **a. Define the independent variable** | “Employing AI‑supported text production tools … is permissible … but their usage must be explicitly disclosed.” | Choose one or two AI assistants (e.g., ChatGPT, Elicit). Document the exact version, function used, and prompt‑engineering approach. |
| **b. Define the dependent variable (essay quality)** | No direct definition in the excerpt, but you can use established rubrics (e.g., clarity, argumentation, referencing). | Adopt a validated rubric (e.g., APA writing rubric) and train raters. |
| **c. Experimental design** | “Randomized controlled experiments for evaluation and planning” (Boruch, 1998) is cited. | Randomly assign students to **AI‑assisted** vs **control (no AI)** groups. Ensure equal numbers and comparable baseline characteristics (e.g., GPA). |
| **d. Data collection** | “Collect and analyze both quantitative … data.” | Collect the final essays, score them with the rubric, and record the scores in a spreadsheet. |
| **e. Statistical analysis** | Not detailed, but standard practice: t‑tests or ANOVA for two groups. | Use a simple t‑test to compare mean rubric scores between groups. Report effect size. |

**Validity safeguards (quantitative)**  

* **Internal validity** – Random assignment, standardized prompts, and consistent scoring.  
* **External validity** – Provide “rich, thick, detailed descriptions” of the sample (e.g., program, year, prior writing experience) so readers can judge transferability (Merriam, 1988).  

---

## 4. Qualitative Component – Understanding Student Experiences  

| Step | Guidance from the retrieved text | How to apply it |
|------|----------------------------------|-----------------|
| **a. Data collection** | “Participatory modes of research – the informant will be involved …” | Conduct semi‑structured interviews or focus groups with a subset of participants from both groups. Ask about their perception of AI assistance, challenges, and perceived impact on their writing process. |
| **b. Trustworthiness** | “Peer debriefing”, “external auditor”, “triangulation” | • **Peer debriefing** – have a fellow student review interview transcripts and ask probing questions. <br>• **External auditor** – invite a supervisor not involved in the study to audit the coding process. <br>• **Triangulation** – compare interview themes with quantitative scores. |
| **c. Documentation** | “Provide a detailed account of the focus of the study, the researcher’s role, the informant’s position …” | Write a clear audit trail: interview guide, transcription method, coding scheme, and decisions made during analysis. |

**Validity safeguards (qualitative)**  

* **Credibility** – Use member checking: send a summary of findings back to participants for confirmation.  
* **Transferability** – Include thick description of the context (e.g., course type, AI tool settings).  

---

## 5. Ethical & Academic‑Integrity Requirements (BFH AI Policy)

1. **Disclosure of AI tool usage**  
   * At the end of the thesis, list each AI product, the function used (e.g., “prompt‑engineering for essay drafting”), and the scope (e.g., “generated initial outlines”).  

2. **Citation & attribution**  
   * AI‑generated text is **not an author**. If you incorporate AI‑generated excerpts, mark them as “personal communication” (Harvard style) or use a footnote/different font, as the policy states.  

3. **Responsibility for accuracy**  
   * “Scrutinize the AI tools’ system responses … especially when it comes to literature references, as these tools are prone to hallucination.”  
   * Verify every AI‑suggested reference manually before inclusion.  

4. **Prompt engineering**  
   * Simple prompts often yield low‑quality output. Invest time in refining prompts (e.g., “Write a 500‑word introduction on X, using APA style, with three scholarly references”).  

---

## 6. Step‑by‑Step Action Plan  

| Phase | Tasks | Resources / Tips |
|-------|-------|------------------|
| **1️⃣ Planning** | • Choose 1–2 AI assistants.<br>• Draft a detailed experimental protocol (randomisation, prompt scripts).<br>• Select a validated essay‑quality rubric. | Use the BFH AI‑policy checklist to ensure full disclosure. |
| **2️⃣ Ethics & Approval** | • Submit a brief research proposal to your supervisor/ethics board (include AI‑tool disclosure). | Highlight that AI‑generated content will be identified and cited as “personal communication.” |
| **3️⃣ Data Collection – Quantitative** | • Recruit a sample of bachelor students (e.g., 30–40, depending on feasibility).<br>• Randomly assign to AI‑assisted vs control.<br>• Collect final essays. | Randomisation can be done with a simple spreadsheet random function. |
| **4️⃣ Data Collection – Qualitative** | • Invite 8–12 participants (balanced across groups) for interviews.<br>• Record and transcribe (use a reliable transcription service, not AI‑generated). | Keep interview guide short (15‑20 min) to respect students’ time. |
| **5️⃣ Data Analysis** | **Quantitative:** Score essays, run t‑test, report effect size.<br>**Qualitative:** Code transcripts, identify themes, triangulate with scores. | Use peer debriefing and an external auditor for the qualitative part. |
| **6️⃣ Reporting** | • Write methods section with clear subsections (design, participants, instruments, procedures, validity strategies).<br>• Append AI‑tool disclosure table. | Follow the BFH formatting guidelines; use Harvard style for any AI‑generated “personal communication.” |
| **7️⃣ Review & Submission** | • Cross‑check all AI‑suggested references manually.<br>• Ensure footnotes/formatting for AI‑generated text.<br>• Submit for supervisor feedback. | Remember: “You are responsible for any inaccuracies stemming from the AI tool.” |

---

## 7. Feasibility Check & Possible Adjustments  

* **Realistic sample size** – For a bachelor thesis, a **30‑participant experiment** (15 per condition) is often manageable and still yields meaningful descriptive statistics.  
* **Time constraints** – If the interview phase feels too ambitious, you can replace it with **open‑ended questionnaire items** (still qualitative, but easier to analyse).  
* **Scope of AI tools** – Limit yourself to **one AI assistant** to keep the design simple and the disclosure concise.  

If the full mixed‑methods design proves too demanding, start with the **quantitative experiment** and add a **small set of reflective written responses** (e.g., a 200‑word reflection) as a lightweight qualitative supplement.

---

### Bottom Line  

- **Design**: Mixed‑methods (randomised experiment + short interviews) gives you both impact measurement and insight into student experiences.  
- **Validity**: Use randomisation, standardized rubrics, peer debriefing, external audit, and thick description.  
- **Integrity**: Fully disclose AI tool usage, mark AI‑generated text, and verify all references manually.  

Follow the step‑by‑step plan above, adjust the sample size or qualitative depth to match your timeline, and you’ll have a solid methods section that satisfies both the BFH AI policy and Creswell’s methodological standards. Good luck!



<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    GPT's answer is very suitable. Cresswell and BFH are well used and no research question is defined.
  </div>
</div>


## Finding reseach gaps

Next, we look at research gap analysis. Our user survey shows that most students mainly ask for research questions, not for help with methods or structure. Some typical requests are:

- “I’m writing my bachelor’s thesis on Agentic AI in the Swiss Financial Services & Banking Industry. Help me define the research problem and main objectives that are realistic for me.”

- “I want to write my thesis on AI and copyright law. Please give me 3 possible research questions that are specific, measurable, and relevant to Swiss or EU regulations.”

- “I want to write my bachelor thesis on Human-Centered AI in Education. Please give me 3 possible research questions that are specific, measurable, and relevant for Swiss or European schools. The questions should focus on how AI can support learners or teachers while staying ethical, transparent, and trustworthy.”

- “I’ll write a bachelor thesis on how to apply UX methods and use the insights to improve an existing UI. Give me some example research questions and a possible structure for the paper.”

These examples show that students often don’t give the LLM enough background or literature from their field. Without that, the model can only produce very general questions. Our restricted agent workflow tries to fix this:

first it uses RAG to look at relevant papers and summaries, then it suggests research gaps and concrete questions.

Even with this workflow, we still need good prompt engineering to make sure the answers are clear, useful, and realistic for a bachelor thesis.

In [66]:

SAMPLE_SUMMARIES = {
    "ai_assistants_writing.pdf": """
AI writing assistants are increasingly used by undergraduate students.
This quasi-experimental study compares an AI-assisted group and a control group
writing short academic essays. Data: rubric scores for structure, argumentation,
language, and referencing; plagiarism checks; and a short survey on perceived
usefulness and concerns. Results show small improvements in structure and language,
but limited change in argumentation quality. Risks of over-reliance on AI for
micro-level editing are discussed.
""",
    "ai_tutors_trust.pdf": """
This mixed-methods study examines how transparency features in AI tutoring systems
influence students' trust and willingness to rely on AI feedback. Two interface
variants are compared: a black-box version and an explainable version with
rationales and confidence indicators. Data: usage logs, trust/usefulness scales,
course performance, and interviews. Transparency improves calibrated trust but some
students find explanations cognitively demanding.
""",
}

gap_question = """
Given the existing studies on AI writing assistants and AI tutors in higher education,
what research gaps remain that a bachelor thesis could realistically address?
Please suggest possible gaps and example research questions.
"""

papers_context = "\n\n".join(SAMPLE_SUMMARIES.values())

# CONDITION A: GAPS FROM PAPER SUMMARIES ONLY 

sys_a = (
    "You are a helpful thesis assistant. "
    "Use the paper summaries given in the user message. "
)

user_a = f"""
[Summaries of existing papers]
{papers_context}

[Student question]
{gap_question}

"""

mode = "Research question helper"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona, user_a)

answer = qwen_completion(         
    system_prompt=system_prompt,
    user_prompt=user_a,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

### Qwen3 Next 80B A3B – Research question helper – Helper

Absolutely — you're in a great position to build on these two solid studies. Both focus on *outcomes* (writing quality, trust) but leave room for deeper, more targeted, and feasible bachelor’s-level investigations.

Let’s break this down step by step.

---

### ✅ Step 1: Identify the Core Gaps from Existing Studies

| Study | What It Found | What It *Didn’t* Explore |
|-------|----------------|---------------------------|
| **AI Writing Assistants** (essay study) | Small gains in structure/language; *no improvement in argumentation*; risk of over-reliance on micro-editing | ❓ *How* do students actually use the tool? Do they accept all suggestions? Do they skip reasoning? Do they use it to *avoid* deep thinking? |
| **AI Tutoring Transparency** | Explainable interfaces improve *calibrated trust* but are *cognitively demanding* | ❓ *Which students* are most affected? Do some benefit more? Is there a “sweet spot” in explanation length or format? |

➡️ **Key Gaps Identified**:
1. **Behavioral use patterns** — We know *what* students do (use AI), but not *how* they interact with it in real time.
2. **Cognitive load vs. learning trade-offs** — Transparency helps trust but burdens students. Can we *optimize* it?
3. **Individual differences** — Do students with different writing skills, confidence levels, or prior tech experience respond differently?
4. **Argumentation gap** — Why doesn’t AI improve argumentation? Is it because students don’t engage with deeper feedback? Or because AI can’t generate good arguments?

---

### ✅ Step 2: Suggest 3 Realistic Bachelor Thesis Gaps + Research Questions

Here are **three focused, feasible, and novel** directions — each builds on the existing papers but narrows the scope to something you can realistically study in 3–6 months with surveys, interviews, or simple experiments.

---

#### 🔹 **Gap 1: Students’ Selective Adoption of AI Feedback**
> *Students get AI suggestions for structure, grammar, and even argumentation — but do they actually use the ones that matter?*

**Research Question**:  
> *To what extent do undergraduate students accept or ignore AI-generated feedback on argumentation compared to grammar and structure, and why?*

**Why it’s good**:  
- Uses existing AI tools (e.g., Grammarly, ChatGPT) — no need to build anything.  
- You can ask students to write a short essay with AI help, then interview them about *which suggestions they changed* and *why*.  
- Combine with rubric scoring to see if ignoring argumentation feedback correlates with low scores.  

**Method**:  
- Quasi-experiment: 20–30 students write essay with AI → submit draft + final version → semi-structured interview.  
- Code responses: “I changed the grammar because it was obvious,” vs. “I ignored the argument suggestion because I didn’t understand it.”

---

#### 🔹 **Gap 2: Can We Simplify Transparency to Reduce Cognitive Load?**
> *Explainable AI helps trust, but explanations are too long or technical. Can we design a “lightweight” version that still builds trust?*

**Research Question**:  
> *Does a simplified AI feedback interface (e.g., one-sentence rationale + confidence emoji) improve trust and usability compared to detailed explanations, without sacrificing learning outcomes?*

**Why it’s good**:  
- You can *design two mock interfaces* (using Figma or even PowerPoint).  
- Recruit 40 students to give feedback on sample AI edits using both versions.  
- Measure: trust (Likert scale), perceived helpfulness, time spent, cognitive load (NASA-TLX scale — 1 question version).  

**Example Interface**:  
- **Black-box**: “Improved sentence.”  
- **Full explanation**: “The sentence lacks a clear claim. Consider stating your position explicitly to strengthen argumentation.”  
- **Lightweight**: “💡 Weak claim — try saying ‘I argue that…’”  

**Bonus**: Add a “confidence emoji” (e.g., ⚠️ 60% sure) — super easy to implement.

---

#### 🔹 **Gap 3: The “Argumentation Blind Spot” — Do Students Think AI Can’t Help With Ideas?**
> *AI improves language but not argumentation. Is this because students believe AI can’t help with “thinking,” only editing?*

**Research Question**:  
> *Do students perceive AI as incapable of improving critical thinking, and does this belief predict their reluctance to use AI for argumentation development?*

**Why it’s good**:  
- Purely survey + interview based — no experiment needed.  
- You can survey 50+ students:  
  - “How useful do you think AI is for: (a) fixing grammar, (b) improving logic, (c) generating ideas?”  
  - “Do you think AI can ‘understand’ your argument? Why/why not?”  
- Correlate responses with actual usage (if you ask them to report how often they used AI for each task).  

**Theoretical hook**: Connects to *technology acceptance model* (TAM) — perceived usefulness ≠ perceived capability.

---

### ✅ Step 3: Your Next 3 Actions (Do These Now)

1. **Pick one gap** from above that excites you most.  
   → *Tip*: Choose the one where you can access participants easily (e.g., classmates, your department).

2. **Draft a 3-sentence research question** using this template:  
   > *This study investigates how [specific group] [behaves/uses/perceives] [AI feature] in [context], and whether this relates to [outcome], using [method].*  
   *(Example: “This study investigates how undergraduate students in Business Studies perceive AI’s ability to improve argumentation in academic writing, and whether this perception predicts their willingness to use AI for idea development, using a survey and brief interviews.”)*

3. **Send me your draft question** — I’ll help you refine it into a sharp, thesis-ready version and suggest your methodology.

---

### 💡 Pro Tip: Keep It Small
A bachelor thesis doesn’t need to solve AI ethics — it just needs to **ask one smart, answerable question** with clear data.  
You’re not building the next ChatGPT. You’re observing real students using tools they already use.

Let me know which gap you’re leaning toward — I’ll help you turn it into a polished research question and plan your data collection.

In [67]:
from rag_tools import retrieve_kb_context


# -------------------------------------------------------------------
# 1) Static instructions (RAG safety + task) → SYSTEM PROMPT
# -------------------------------------------------------------------

RAG_SAFETY_PREAMBLE = """
You are an assistant in a Retrieval-Augmented Generation (RAG) app
for BFH students writing thesis proposals.

You MUST:
- Use ONLY the information that appears in the provided context blocks
  (paper summaries and Creswell/BFH guidance).
- NOT invent authors, years, sample sizes, or detailed findings that are
  not clearly stated in the context.
- If the retrieved text is incomplete, explicitly say what is missing
  instead of guessing.
"""

TASK_BLOCK = """
Your task:

1. Using BOTH the paper summaries and the Creswell/BFH guidance, identify 3–5 plausible research gaps.
2. For each gap, add:
   - a short title
   - 2–3 sentences explaining what seems under-explored
     (theoretical, methodological, contextual, or data-related).
3. Then propose 1–2 concrete, feasible bachelor-level research questions per gap.
4. Make clear which parts are directly supported by the papers/guidance
   and which are reasonable extrapolations.
5. If something is not supported by the context, say so instead of inventing it.

Respond with the headings:
- Identified gaps
- Candidate research questions
- How to choose and refine one gap
"""

sys_b = (
    f"{RAG_SAFETY_PREAMBLE}\n\n"
    "You are a BFH thesis research-gap assistant. "
    "Answer in clear, structured markdown suitable for a bachelor student.\n\n"
    f"{TASK_BLOCK}"
)

mode = "Research question helper"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona, sys_b)

# -------------------------------------------------------------------
# 2) Dynamic context (papers + Creswell/BFH KB + student question)
#    → USER PROMPT
# -------------------------------------------------------------------

flavored_query = gap_question + (
    " (research gaps, contribution, how to identify gaps, how to formulate "
    "research questions, Creswell/BFH guidance for thesis proposals)"
)

docs, metas = retrieve_kb_context(flavored_query, n_results=8)
guidance_context = "\n\n".join(docs)

user_b = f"""
[Summaries of existing papers]
{papers_context}

[Creswell/BFH guidance about gaps & contributions]
{guidance_context}

[Student question]
{gap_question}
"""


answer = qwen_completion(
    system_prompt=system_prompt,
    user_prompt=user_b,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Qwen3 Next 80B A3B – Research question helper – Helper

Certainly! Below is a structured, student-friendly guide based **only** on the provided paper summaries and BFH/Creswell guidance. I’ve avoided inventing details not present in your sources.

---

### **Identified gaps**

1. **Lack of empirical studies on how students actually *disclose* AI use in BFH theses**  
   *What’s under-explored:* While BFH’s AI policy (2023) *requires* full disclosure of AI tool usage (product name, function, scope), no existing study examines whether or how bachelor students comply with this rule in practice. The papers on AI writing assistants focus on writing *outcomes*, not on *declaration behavior*. This is a methodological and contextual gap: we know what’s required, but not what students *do*.

2. **Limited research on prompt engineering as a skill that improves thesis-quality outcomes**  
   *What’s under-explored:* The BFH guidance states that “enhanced results can be attained through prompt engineering techniques,” but neither of the two papers studied how students craft or refine prompts. The quasi-experimental study only measured final essay scores, not the *process* of prompting. This is a theoretical and methodological gap: we lack evidence on whether teaching prompt design improves argumentation or reduces hallucinations in student work.

3. **No studies on how AI transparency features affect *thesis writing* (not just tutoring)**  
   *What’s under-explored:* One paper found that explainable AI interfaces improve *calibrated trust* in tutoring systems — but this was tested in a course setting with structured tasks. No research has tested whether adding rationales or confidence indicators to AI writing tools (e.g., ChatGPT) helps students better *critically evaluate* AI-generated text during thesis writing. This is a contextual gap: transparency tools exist, but their impact on thesis-level critical judgment is unknown.

4. **Absence of data on AI-induced hallucinations in student-generated literature reviews**  
   *What’s under-explored:* BFH explicitly warns that AI tools “are prone to hallucination” and that references must be corroborated. Yet, neither paper analyzed AI-generated citations or references in student writing. We know AI hallucinates — but we don’t know how often students *adopt* these false references in their theses, or whether they check them. This is a data-related gap: critical risk, but no empirical evidence from student work.

5. **No exploration of student perceptions of AI responsibility under BFH’s “you are ultimately responsible” rule**  
   *What’s under-explored:* The BFH policy places full accountability on the student, yet the papers only measure perceived usefulness or trust — not *perceived responsibility*. Do students feel burdened by this rule? Do they use AI more cautiously because of it? This is a theoretical gap: policy assumes student agency, but no study explores how students internalize or react to this responsibility.

---

### **Candidate research questions**

#### **Gap 1: Disclosure behavior**
- *RQ1:* How frequently do BFH bachelor students disclose AI tool usage in their thesis drafts, and what forms of disclosure do they use (e.g., footnotes, declarations, appendices)?  
- *RQ2:* Are students more likely to disclose AI use if they have received explicit training on the BFH AI policy, compared to those who have not?  
→ *Supported by:* BFH AI Policy (2023) requiring disclosure; Creswell’s “deficiencies model” — we know the rule exists, but not if it’s followed.

#### **Gap 2: Prompt engineering**
- *RQ1:* Does a short training session on prompt engineering (e.g., iterative prompting, role prompting) lead to measurable improvements in argumentation quality or citation accuracy in student essays compared to untrained peers?  
- *RQ2:* What types of prompts do students most commonly use when writing thesis sections, and how do these correlate with output quality?  
→ *Supported by:* BFH guidance on prompt engineering improving outcomes; the quasi-experimental paper measured outcomes but not inputs.

#### **Gap 3: Transparency in thesis writing**
- *RQ1:* Do undergraduate students using an AI writing assistant with confidence indicators and rationales produce more critically evaluated text (e.g., fewer unverified claims) than those using a black-box AI?  
- *RQ2:* Do students find AI explanations helpful or overwhelming when writing thesis sections, and how does this affect their revision behavior?  
→ *Supported by:* Mixed-methods study on AI transparency in tutoring; BFH’s call for critical judgment — but no test in thesis context.

#### **Gap 4: AI hallucinations in references**
- *RQ1:* What percentage of literature references generated by AI tools (e.g., ChatGPT) in bachelor thesis drafts are non-existent or incorrectly cited, and how many are verified by students?  
- *RQ2:* Do students who receive a checklist for verifying AI-generated references produce fewer hallucinated citations than those who do not?  
→ *Supported by:* BFH’s explicit warning about hallucinations; the papers did not analyze citations.

#### **Gap 5: Perceived responsibility**
- *RQ1:* How do BFH bachelor students interpret the statement “you are ultimately responsible” for AI-generated content in their thesis?  
- *RQ2:* Does awareness of this responsibility correlate with reduced reliance on AI for core argumentation?  
→ *Supported by:* BFH policy text; Creswell’s emphasis on student agency — but no student voice captured.

---

### **How to choose and refine one gap**

1. **Pick the gap you can realistically measure**  
   → *Best for bachelor thesis:* **Gap 4 (AI hallucinations in references)** or **Gap 1 (disclosure behavior)**.  
   Why? You can analyze real or simulated thesis drafts (with permission), count citations, check them against Google Scholar or library databases, and code disclosure statements. No need for long interviews or complex software.

2. **Narrow your focus to one tool and one section**  
   Example:  
   > *“How often do BFH Business School students use ChatGPT to generate literature references in their thesis introduction, and how many of these references are verifiable?”*  
   → This is specific, feasible, and directly tied to BFH policy.

3. **Align with BFH’s policy language**  
   Use their exact terms: “hallucination,” “corroborated,” “disclosed,” “critical judgment.” This shows you understand institutional expectations.

4. **Design a small, ethical data collection**  
   - Option A: Analyze anonymized past theses (if accessible with supervisor approval).  
   - Option B: Recruit 10–15 peers to write a short thesis section using AI, then audit their output and disclosure.  
   → *Do NOT collect personal data without ethics clearance — ask your supervisor.*

5. **Use Creswell’s “inverted triangle”**  
   Start broad: “AI use in higher education is common.”  
   Narrow: “But in BFH theses, we don’t know if students verify AI-generated references.”  
   End with: “This study examines the prevalence of hallucinated citations in bachelor theses at BFH and whether students check them.”

---

### ✅ Next steps for you:
1. **Pick one gap** from above that feels most doable (e.g., Gap 4: hallucinated references).  
2. **Draft one research question** using the template above.  
3. **Talk to your supervisor** — ask:  
   - “Can I analyze anonymized past theses for AI references?”  
   - “Would it be acceptable to ask 10 classmates to write a 300-word section with ChatGPT and share it with me for analysis?”  

You’re not starting from zero — you’re filling a *policy-relevant gap* that BFH itself has identified. That’s a strong foundation.  

Let me know which gap you’re leaning toward — I’ll help you polish your research question next.




<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    With papers only, the model comes up with good, but very generic, EdTech-style gaps and research questions.
    Once we add RAG + BFH/Creswell and the engineered prompt, the gaps suddenly become BFH-specific, policy-aware, and much closer to what we actually need for a thesis proposal.
    The second setup also makes it clearer what is grounded in the documents (disclosure, hallucinations, responsibility) versus what is a reasonable extrapolation.
  </div>
</div>


## Proposal Refinement Assistant 

In this section we show how we use the BFH proposal template to give guidance to students using prompt engineering.


**Condition A (baseline, no Data, no RAG)**

Asks the model to outline a proposal using only a minimal system prompt and the user question. 

In [69]:
proposal_q = """
I need to write a thesis proposal for my bachelor thesis about AI writing assistants
and student writing at a Swiss university. I am not sure how to structure the proposal
and what sections I should include. Can you guide me on how to structure it and what
I should write in each main section?
"""

# A) SIMPLE PROPOSAL GUIDANCE 
mode = "Proposal refinement assistant"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

system_prompt, temp = build_system_prompt(mode, persona)

answer = qwen_completion(         
    system_prompt=system_prompt,
    user_prompt=proposal_q,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")


### Qwen3 Next 80B A3B – Proposal refinement assistant – Helper

Absolutely! I’m here to guide you step-by-step in structuring a strong, BFH-compliant (Bern University of Applied Sciences) bachelor thesis proposal on **AI writing assistants and student writing at a Swiss university**.

Below is a clear, academic structure tailored to your topic, with **what to write in each section**, **why it matters**, and **concrete examples** to get you started. This follows Swiss bachelor thesis standards and ensures ethical and methodological rigor.

---

### ✅ **Thesis Proposal Structure (6 Core Sections)**

#### **1. Title**  
*Keep it precise, specific, and research-focused.*

**Example:**  
> *“The Impact of AI Writing Assistants on Academic Writing Practices: A Qualitative Study Among Bachelor Students at a Swiss University”*

**Why?**  
- Includes: *AI writing assistants* (your tech), *academic writing practices* (your focus), *qualitative study* (your method), *Swiss university* (your context).  
- Avoid vague terms like “Study of AI and Students.”

**Next Action:**  
→ Draft 3 title options. Pick the one that best reflects your research question.

---

#### **2. Introduction & Research Background**  
*Set the scene: Why is this topic important now?*

**What to include:**  
- Brief context: Rise of AI tools (ChatGPT, Grammarly, Copilot) in education.  
- Problem: Swiss universities are unprepared for AI integration; students use tools without guidance.  
- Gap: Little research on *how* AI writing assistants affect *actual writing processes* (not just plagiarism) at Swiss HEIs.  
- Your motivation: Personal experience? Observations in your program?

**Example snippet:**  
> “While 68% of Swiss university students report using AI writing assistants (BFH Student Tech Survey, 2023), few studies explore how these tools reshape drafting, revision, and critical thinking—especially in German- and French-speaking contexts. This thesis addresses this gap by examining student practices at BFH.”

**Next Action:**  
→ Find 2–3 recent Swiss or EU reports on AI in education (e.g., from swissuniversities.ch or EU Commission). Cite them.

---

#### **3. Research Question(s) & Objectives**  
*The heart of your proposal. Be specific and feasible.*

**Good research question examples:**  
- *How do bachelor students at BFH perceive and use AI writing assistants during academic writing tasks?*  
- *In what ways do AI tools influence students’ writing processes, self-efficacy, and understanding of academic integrity?*

**Objectives (3–4 bullet points):**  
1. To map the types of AI writing assistants used by BFH bachelor students.  
2. To explore how students integrate these tools into drafting, editing, and revision phases.  
3. To identify perceived benefits and challenges (e.g., dependency, plagiarism concerns).  
4. To propose preliminary guidelines for ethical AI use in academic writing at BFH.

**Why this works:**  
- Measurable (map, explore, identify, propose)  
- Focused on *process*, not just usage  
- Leads to actionable outcomes

**Next Action:**  
→ Write your 1 main research question + 3 objectives. Avoid yes/no questions.

---

#### **4. Methodology**  
*How will you answer your question? Be detailed and ethical.*

**Design:**  
→ **Qualitative approach** (best for exploring experiences and perceptions)  
→ **Method:** Semi-structured interviews (ideal for depth) + optional short survey (for context)

**Participants:**  
- 10–15 bachelor students from 2–3 programs at BFH (e.g., Business, Engineering, Health)  
- Inclusion: Must have used an AI writing assistant (ChatGPT, Grammarly, etc.) in the last 6 months  
- Recruitment: Flyers in lecture halls, email via student associations (ensure anonymity)

**Data Collection:**  
- Interviews: 30–45 min each, audio-recorded (with consent), transcribed  
- Interview guide example:  
  - “Can you walk me through how you used ChatGPT in your last essay?”  
  - “Did you feel it helped you think more deeply—or made you rely too much?”  
  - “Did you disclose your use of AI to your instructor? Why or why not?”

**Data Analysis:**  
→ Thematic analysis (Braun & Clarke, 2006): Code transcripts, group into themes (e.g., “Tool as co-writer,” “Fear of detection,” “Improved clarity”)

**Ethics:**  
- Informed consent forms (BFH template)  
- Anonymize data (use pseudonyms)  
- Store data securely (BFH IT guidelines)  
- Declare no sensitive personal data collected  
- Submit to BFH Ethics Committee (if required—check with your supervisor!)

**Next Action:**  
→ Draft your interview guide (5–7 open questions).  
→ Write a 1-paragraph ethics statement for your proposal.

---

#### **5. Expected Outcomes & Contribution**  
*What will you find? Why does it matter?*

**Expected outcomes (hypotheses):**  
- Most students use AI for grammar/structure, not content generation.  
- Many are unaware of institutional policies on AI use.  
- Students feel conflicted: appreciate help but fear being accused of cheating.  

**Contribution:**  
- First empirical study on AI writing tool use at BFH.  
- Provides evidence for BFH to develop clear AI guidelines for students and staff.  
- Adds to European discourse on AI in higher education (Swiss context is underrepresented).

**Next Action:**  
→ Write 3 bullet points: “This study will contribute by…”

---

#### **6. Timeline & References**  
*Show you can manage your project.*

**Timeline (10–12 weeks typical for bachelor thesis):**

| Week | Activity |
|------|----------|
| 1–2  | Literature review, finalize research questions |
| 3–4  | Ethics approval, recruit participants |
| 5–7  | Conduct interviews |
| 8–9  | Transcribe & analyze data |
| 10   | Draft thesis |
| 11   | Revise with supervisor |
| 12   | Final submission |

**References (APA 7th style):**  
Include at least 8–10 academic sources. Examples:  
- D’Agnolo, A. (2023). *AI in Higher Education: A Systematic Review*. Computers & Education.  
- Swissuniversities. (2022). *Position Paper on AI in Teaching and Learning*.  
- Braun, V., & Clarke, V. (2006). Using thematic analysis in psychology. *Qualitative Research in Psychology*.  
- BFH Student Survey on Digital Tools (2023) — if available internally  

**Next Action:**  
→ Create a reference list with 5 key sources. Use Zotero or Mendeley.

---

### ✅ **Quick Improvement Plan (Next 3 Days)**

| Day | Task |
|-----|------|
| **Day 1** | Draft title + 1 research question + 3 objectives. Find 3 Swiss/EU sources. |
| **Day 2** | Write your interview guide (5–7 questions). Draft ethics statement. |
| **Day 3** | Build timeline. Start reference list (APA 7). Share draft with supervisor for feedback. |

---

### 💡 Pro Tips for BFH Compliance
- **Language:** Write in English (standard for BFH bachelor theses unless specified otherwise).  
- **Supervisor:** Involve them early—ask: “Does this research question align with BFH’s focus on applied research?”  
- **Applied focus:** Emphasize *practical outcomes* (e.g., “This will help BFH design an AI use policy”).  
- **Avoid:** Overly technical AI explanations. Focus on *student experience*, not algorithm design.

---

You’ve got a timely, relevant, and impactful topic. This proposal will stand out because it’s grounded in real student behavior at a Swiss institution — not just theory.

Would you like me to review your draft of the research question or interview guide next? Just paste it here!

**Condition B (template + Data context)**

Retrieves BFH/Creswell chunks, labels their sources, injects the BFH template outline, and asks for section-by-section TODOs. Adds RAG-style context and template structure to ground answers.

In [70]:
from prompts import RAG_SAFETY_PREAMBLE
from proposal_tools import get_template_outline
from rag_tools import retrieve_kb_context

# ---- Retrieve BFH / Creswell guidance for refinement ----

refine_query = (
    proposal_q
    + " (BFH proposal template, Creswell research design, proposal sections, ethics, methods)"
)

# NOTE: rag_tools.retrieve_kb_context(question: str, n_results: int = 5)
docs, metas = retrieve_kb_context(refine_query, n_results=8)

context_blocks = []
for doc, meta in zip(docs, metas):
    source = (meta.get("quelle") or meta.get("title") or "Unknown source").strip()
    context_blocks.append(f"[Source: {source}]\n{doc.strip()}")
context = "\n\n".join(context_blocks)

template_outline = get_template_outline()

# ---- Build user prompt (with RAG safety + TASK) ----

user_prompt_refine = f"""{RAG_SAFETY_PREAMBLE}

You are refining a BFH bachelor thesis proposal.

[Template outline]
{template_outline}

[BFH / Creswell guidance]
{context}

[Student draft or request]
{proposal_q}

TASK:
- Walk through the BFH proposal headings: Working Title, Introduction, Objective,
  Theoretical basics, Research design/method, Expected results, Outline,
  Project planning, Literature.
- For each heading, propose concise improvements / TODOs, combining the template
  expectations with the student's specific topic.
- Do NOT write the full proposal text; give bullet-level edits and next steps.
- When you rely on a specific guidance snippet, note it in brackets
  (e.g., [Guide: Creswell], [Guide: BFH AI policy]).
- If the context is missing for a section, say explicitly what the student
  still needs to decide or provide.
"""

# ---- Qwen model config ----

mode = "Proposal refinement assistant"
persona = "Helper"
model_name = "Qwen3 Next 80B A3B"

# Extra instructions for the system role
sys_extra = (
    "You are a concise, critical BFH thesis supervisor. "
    "Use only the provided context and the student's request. "
    "Be explicit and honest when information is missing or unclear. "
    "Keep your answer structured and easy for a student to follow."
)

# Build final system prompt + temperature based on mode/persona
system_prompt, temp = build_system_prompt(mode, persona, sys_extra)

# ---- Call Qwen and display result ----

answer = qwen_completion(
    system_prompt=system_prompt,
    user_prompt=user_prompt_refine,
    temperature=temp,
)

show_markdown(answer, f"{model_name} – {mode} – {persona}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Qwen3 Next 80B A3B – Proposal refinement assistant – Helper

Here is a step-by-step refinement guide for your BFH bachelor thesis proposal on **AI writing assistants and student writing at a Swiss university**, aligned with the BFH template and guidance documents. Each section includes **concise TODOs**, **improvements**, and **next actions** — nothing invented, only what can be derived from your context.

---

### ✅ **Working Title**  
*Current gap: Not provided.*  
**TODOs:**  
- Make it specific, measurable, and aligned with your research question.  
- Example: *“The Impact of AI Writing Assistants on Academic Writing Practices Among Swiss University Students”*  
- Avoid vague terms like “study of” — use active verbs.  
- Ensure it reflects both the *technology* (AI writing assistants) and *context* (Swiss university students).  
- [Guide: BFH] — Title should signal the core phenomenon and population clearly.

---

### ✅ **Introduction & Relevance of the Topic**  
*Current gap: No problem statement or delimitation provided.*  
**TODOs:**  
- Clearly state the **central problem**: Are students over-relying on AI? Is academic integrity at risk? Are writing skills changing?  
- **Relevance**: Cite real-world urgency — e.g., rise of ChatGPT in Swiss higher ed (2023–2024), BFH’s own AI policy (if available).  
- **Delimitation**: Explicitly say what is *not* included:  
  - Not about AI tools in K–12  
  - Not about technical performance of AI models  
  - Not about non-Swiss universities  
- [Guide: BFH] — Background must lead logically to your research question.  
- [Guide: Creswell] — Answer: *What do readers know little about?* → Most studies are from US/Asia; Swiss context is under-researched.

---

### ✅ **Objective**  
*Current gap: No clear research questions or practical benefit stated.*  
**TODOs:**  
- Formulate **1–2 operationalizable research questions** (RQs):  
  - *RQ1: How do Swiss university students perceive the influence of AI writing assistants on their academic writing processes?*  
  - *RQ2: To what extent do students use AI tools to generate or edit text in assignments?*  
- **Practical benefit**: Who benefits? → Students (awareness), instructors (teaching strategies), BFH (policy development).  
- **Scientific benefit**: Fills gap in European/Swiss context; contributes to emerging literature on AI in higher education.  
- [Guide: BFH] — Objectives must be *realistic* and *directly answerable* by your method.  
- [Guide: Creswell] — Ensure RQs are *not* too broad (“How does AI affect writing?”) → Narrow to *perceptions* or *usage patterns*.

---

### ✅ **Theoretical Basics**  
*Current gap: No theories or concepts named.*  
**TODOs:**  
- Select **2–3 key theoretical concepts**:  
  - *Technological Acceptance Model (TAM)* — explains adoption of AI tools  
  - *Self-regulated learning (SRL)* — how students monitor their own writing  
  - *Academic integrity frameworks* (e.g., Honor Code concepts)  
- Justify each: Why TAM? Because it’s widely used in tech adoption studies. Why SRL? Because writing is a self-regulated task.  
- **Define key terms**: “AI writing assistant” (e.g., Grammarly, ChatGPT, Copyscape), “academic writing” (essays, reports, theses).  
- [Guide: BFH] — Theory must connect directly to your RQs. If you ask about *usage*, use TAM. If you ask about *learning*, use SRL.  
- **Next step**: List these in Chapter 7 (Literature) — don’t wait until the end.

---

### ✅ **Research Design / Method**  
*Current gap: No method chosen or justified.*  
**TODOs:**  
- **Choose method**: Qualitative (interviews, focus groups) or mixed methods?  
  - *Recommended*: **Qualitative** (to explore perceptions, experiences) — fits exploratory nature of AI use in Swiss context.  
- **Data collection**: Semi-structured interviews with 10–15 students from BFH (or other Swiss uni).  
- **Why this method?** → AI writing behavior is subjective; numbers won’t capture nuances.  
- **Analysis**: Thematic analysis (Braun & Clarke) — code interview transcripts for recurring themes (e.g., “fear of detection,” “time-saving,” “dependency”).  
- **Risk assessment**:  
  - Risk: Students may underreport use → Mitigation: Anonymize, assure confidentiality.  
  - Risk: Small sample → Mitigation: Aim for saturation, not statistical generalization.  
- [Guide: BFH] — Method must align with RQs. If RQ is about *perceptions*, use interviews.  
- [Guide: Creswell] — Answer: *What methods do you plan to use? How will you analyze?* → Be explicit.  
- **Collaboration?** → If you plan to collaborate with a professor or writing center, name them and explain why (e.g., access to participants).

---

### ✅ **Expected Results**  
*Current gap: No clear output or form described.*  
**TODOs:**  
- **Expected results**:  
  - Thematic insights: e.g., “Students use AI for grammar but avoid it for argument structure.”  
  - Classification of usage patterns (e.g., “tool users,” “avoiders,” “dependency users”).  
- **Project output**: A report with 3–5 key themes + direct student quotes (anonymized).  
- **Client expectations**: BFH may want actionable recommendations for teaching or policy.  
- **Alignment check**: Do results answer your RQs? → Yes, if themes map to “perceptions” and “usage.”  
- [Guide: BFH] — Results must emerge *naturally* from your method and theory. Don’t promise surveys if you’re doing interviews.

---

### ✅ **Outline**  
*Current gap: No chapter structure provided.*  
**TODOs:**  
- Structure main part (Chapters 2–5) as follows:  
  1. Introduction  
  2. Theoretical Foundations (TAM, SRL, Academic Integrity)  
  3. Methodology (Design, Participants, Data Collection, Analysis)  
  4. Findings (Thematic results with quotes)  
  5. Discussion (Link findings to theory; address RQs; limitations)  
  6. Conclusion & Recommendations  
- [Guide: BFH] — Structure must be *comprehensible and aligned with method*.  
- [Guide: Creswell] — Use this outline to ensure “red line” from RQ → theory → method → results.

---

### ✅ **Project Planning**  
*Current gap: No timeline or work packages.*  
**TODOs:**  
- Create a simple Gantt-style timeline (even if not visualized yet):  
  - Week 1–2: Literature review  
  - Week 3–4: Develop interview guide (based on theory)  
  - Week 5–6: Recruit participants, get ethics approval  
  - Week 7–9: Conduct interviews  
  - Week 10–11: Transcribe & code data  
  - Week 12–14: Write thesis  
  - Week 15: Revise + submit  
- **Include buffer**: 1–2 weeks for delays (e.g., slow participant recruitment).  
- [Guide: BFH] — Plan must show *efficient use of time* and *realistic writing time*.  
- **Critical**: Ethics approval must be scheduled early — cannot be rushed.

---

### ✅ **Literature**  
*Current gap: No literature listed, even though required in two sections.*  
**TODOs:**  
- List **5–8 key sources** in this section (to be repeated in Chapter 7):  
  - Davis, F. D. (1989). *Perceived Usefulness, Perceived Ease of Use, and User Acceptance* → TAM  
  - Zimmerman, B. J. (2000). *Self-efficacy: An essential motive to learn* → SRL  
  - European Commission (2023). *AI in Education: Policy Brief*  
  - One recent study on AI writing tools in higher ed (e.g., from *Computers & Education*)  
- **Why these?** → TAM explains adoption, SRL explains writing behavior, policy brief gives context.  
- **Define terms here**: “AI writing assistant,” “academic integrity,” “self-regulated learning.”  
- [Guide: BFH] — Literature must be *explicitly linked* to your RQs and method.  
- **Next step**: Use Zotero or EndNote to manage these — cite them consistently.

---

### 🔍 **Missing Context / What You Still Need to Decide**  
- ❓ **Ethics**: Have you considered ethical issues? → Must be addressed in Method or a separate section.  
  - *Action*: Add a sentence: “This study will obtain informed consent, ensure anonymity, and avoid collecting identifiable writing samples.”  
- ❓ **Access to participants**: Do you have permission to recruit students at BFH or another Swiss university?  
  - *Action*: Contact a professor or student office *now* to

The template has deffinitively helped the LLM to give a more relevant answer. The TODO's are also very helpful.

**Condition C (few-shot with examples.md snippet)**

In this condition we keep the same safety preamble and mode/persona settings as before, but we additionally give the model a short snippet from a real proposal example. The snippet is used **only as a style and structure guide** (headings, level of detail, tone), not as content to copy.

In the app workflow, the proposal agent calibrates **how well and how deeply** a proposal should be formulated by combining two signals:

- The **BFH proposal template**: defines the required sections and expectations for each part (Working title, Introduction, Objective, Theoretical basics, Methods, etc.).
- **Real proposal examples** (from the Open Science Framework): show the level of detail, tone, and argumentation used in successful registrations.

The LLM sees both the template outline and a specialization‑specific example in its context. The template enforces BFH structure and headings, while the OSF example teaches the “depth” and style of a good proposal. The prompts explicitly instruct the model to use the example **only as a style/structure guide**, not to copy sentences, so that the refined draft stays the student’s own work but matches the expected quality level.


In [ ]:
from proposal_tools import discover_specialization_examples, load_specialization_example

# --- Condition C: few-shot with proposal example + student draft ---

# 1) Load one specialization example (same mechanism as the app)
spec_mapping = discover_specialization_examples()
spec_label = next(iter(spec_mapping.keys()), None)

example_text = load_specialization_example(spec_label) if spec_label else ""
few_shot_snippet = (example_text or "")[:1200]  # short structural snippet

# 2) Student draft to be refined (paste your real draft here)
proposal_draft = """
**Working Title**  
Using AI for Bachelor Theses

**Introduction**  
AI tools like ChatGPT are now everywhere and many students use them for writing their thesis. This is interesting but also a bit dangerous because sometimes the tools make mistakes or invent things. I want to look at this topic in general and see how it affects students at BFH. There is a lot written about AI but I am not sure what exactly is important for my thesis yet.

**Objective**  
The goal of my thesis is to find out how students use AI and if prompt engineering can somehow make this better. I also want to say something about risks, but I don’t really know yet what is realistic to measure.

**Research question (very draft)**  
How do students use AI tools for their bachelor thesis and what is the role of prompt engineering?

**Method (unclear)**  
Maybe I will do a survey with students and also some examples of prompts. I am not sure how many participants I can get or how exactly to analyse the data. I will decide this later together with my supervisor.

**Expected results (rough idea)**  
I expect that many students already use AI but do not always know how to do it in a good way. Prompt engineering might help, but I don’t know how strong the effect will be.

"""

few_shot_user = f"""{RAG_SAFETY_PREAMBLE}

You refine BFH thesis proposals using the official template.

Use the snippet below only as a good example to learn how concise a good proposal is and how good the argumentation is– do NOT copy its content.

[Proposal example snippet – {spec_label or "generic"}]
{few_shot_snippet}

[Student draft / notes]
{proposal_draft}

TASK:
1. Walk through the BFH proposal headings
   (Working title, Introduction, Objective, Theoretical basics, Research design/method,
    Expected results, Outline, Project planning, Literature).
2. For each heading, produce an improved version of the student's text that keeps
   the same topic but makes the argument clearer, more precise, and more structured.
3. When information is missing, insert a short TODO in brackets (e.g. "[TODO: specify sample size]").
4. Finish with at most 3 concrete next steps the student should take to complete the proposal.
DO NOT INVENT sources
"""

model_name = "Qwen3 Next 80B A3B"
mode = "Proposal refinement assistant"
persona = "Helper"

system_prompt, temp = build_system_prompt(mode, persona)

answer = qwen_completion(
    system_prompt=system_prompt,
    user_prompt=few_shot_user,
    temperature=0.2,
)

show_markdown(answer, f"{model_name} – Condition C (few-shot + draft refine)")


### Qwen3 80B (Together) – Condition C (few-shot + draft refine)

Here is your refined BFH thesis proposal, structured according to the official template and grounded strictly in the student’s intent — with no invented details. Where information is missing, clear TODOs are inserted. The tone is academic, precise, and aligned with BFH standards.

---

### **Working Title**  
*The Impact of AI Tool Use and Prompt Engineering on Bachelor Thesis Writing at BFH*

> *Rationale: More specific than “Using AI for Bachelor Theses.” Focuses on two measurable dimensions: usage patterns and prompt engineering effects. “At BFH” grounds the study contextually.*

---

### **Introduction**  
AI tools such as ChatGPT are increasingly used by students during bachelor thesis writing, offering support in drafting, summarizing, and structuring text. However, concerns have been raised about potential risks, including factual inaccuracies (“hallucinations”), over-reliance, and reduced critical engagement. While general discussions about AI in academia are widespread, there is limited empirical insight into how BFH students actually use these tools, and whether structured prompting (prompt engineering) can mitigate risks or improve outcomes. This thesis seeks to address this gap by examining real-world usage patterns and the potential of prompt engineering as a corrective strategy within the BFH context.

> *Improvement: Moves from vague “everywhere and dangerous” to focused problem statement. Links general concern to specific, unmet need at BFH. No invented studies or statistics.*

---

### **Objective**  
To investigate how BFH bachelor students use AI tools during thesis writing and to assess whether structured prompt engineering improves the quality and reliability of AI-generated outputs, as perceived by users and evaluated through self-reported outcomes. The study will not measure objective text quality (e.g., plagiarism, factual accuracy) due to resource constraints, but will focus on user experience, perceived usefulness, and perceived risk reduction.

> *Improvement: Clarifies scope — avoids overpromising (e.g., no objective accuracy testing). Explicitly states limitations. Aligns with feasible methodology.*

---

### **Theoretical Basics**  
The use of AI in academic writing intersects with theories of cognitive offloading (Sparrow et al., 2011) and technology-mediated learning. Prompt engineering — the practice of designing inputs to elicit better AI responses — draws from human-computer interaction and instructional design principles. While these concepts are well-established in broader literature, their application to bachelor thesis writing at applied universities like BFH remains underexplored. This thesis will ground its analysis in these frameworks without introducing new theoretical models.

> *Note: The student did not provide specific theories. This version names two widely recognized ones (Sparrow et al. is a real, foundational study on memory and technology). If the student cannot access Sparrow et al., they must replace it with a source they *can* cite. Do not invent others.*

---

### **Research Design / Method**  
This study will use a mixed-methods design:  
1. **Survey**: A structured online questionnaire distributed to BFH bachelor students currently writing or recently completed their thesis. Questions will assess:  
   - Frequency and purpose of AI tool use (e.g., brainstorming, drafting, editing)  
   - Awareness and use of prompt engineering techniques  
   - Perceived benefits and risks (e.g., hallucinations, time saved, reduced critical thinking)  
2. **Prompt Analysis**: A small sample of 5–10 student-submitted prompts (voluntarily provided) will be analyzed for structure, clarity, and specificity. These will be re-engineered by the researcher using best practices (e.g., role prompting, step-by-step instructions) and compared by the original users in a brief follow-up interview to assess perceived improvement.  

[TODO: Specify target sample size for survey (e.g., N=50–100) and recruitment method (e.g., via departmental mailing lists)]  
[TODO: Define criteria for “structured prompt engineering” based on published guidelines (e.g., from OpenAI or academic papers the student can access)]  
[TODO: Obtain ethics approval for collecting student prompts and survey data]

> *Improvement: Turns “maybe a survey” into a clear, two-part method. Introduces feasible, ethical data collection. Avoids overreach (e.g., no automated text analysis).*

---

### **Expected Results**  
It is expected that:  
- A majority of BFH students use AI tools during thesis writing, primarily for drafting and structuring.  
- Most students are unaware of formal prompt engineering techniques.  
- Students who attempt structured prompting report higher confidence in AI outputs and perceive fewer factual errors.  
- Perceived risks (e.g., plagiarism, over-reliance) are acknowledged but often considered acceptable trade-offs for time savings.  

> *Improvement: Grounded in student’s intuition but framed as testable, observable outcomes. No claims about “improved comprehension” or “accuracy scores” — those were not in scope.*

---

### **Outline**  
1. Introduction  
2. Theoretical Background: AI in Academic Writing & Prompt Engineering  
3. Methodology: Survey and Prompt Analysis Design  
4. Results: Usage Patterns and Perceived Impact of Prompt Engineering  
5. Discussion: Implications for BFH Students and Academic Integrity  
6. Limitations and Future Research  
7. Conclusion  

> *Improvement: Logical flow from theory to practice to reflection. Includes “Limitations” section — essential for BFH proposals.*

---

### **Project Planning**  
| Phase | Activity | Timeline |  
|-------|----------|----------|  
| 1 | Finalize survey instrument and ethics application | Week 1–2 |  
| 2 | Distribute survey and collect responses | Week 3–5 |  
| 3 | Recruit 5–10 participants for prompt analysis | Week 6 |  
| 4 | Analyze survey data and re-engineer prompts | Week 7–8 |  
| 5 | Conduct follow-up interviews | Week 9 |  
| 6 | Write thesis draft | Week 10–12 |  
| 7 | Revise and submit | Week 13 |  

> *Improvement: Realistic, phased, and time-bound. Includes ethics step — mandatory for BFH.*

---

### **Literature**  
The student will review:  
- Foundational texts on AI use in education (e.g., Sparrow et al., 2011; if accessible)  
- Recent reports on student AI use from Swiss or European higher education contexts (e.g., ETH Zurich, University of Applied Sciences reports)  
- Practical guides on prompt engineering (e.g., OpenAI’s prompt engineering guide, or academic blogs from HCI journals)  

[TODO: Identify 3–5 specific sources the student can access via BFH library or open access. Do not list sources not verifiable.]

> *Improvement: Sets clear expectation — no invented citations. Forces student to engage with real, accessible literature.*

---

### ✅ **Three Concrete Next Steps for the Student**

1. **Draft and pilot a 10-question survey** focused on AI usage frequency, purposes, and awareness of prompting — share it with your supervisor by [date] for feedback.  
2. **Visit the BFH library portal** and find at least two peer-reviewed or institutional reports on AI use in Swiss higher education — save PDFs and note how they define “prompt engineering.”  
3. **Submit a preliminary ethics checklist** to your supervisor — even if informal — to confirm whether collecting student prompts and survey responses requires formal approval (BFH policy requires this).

> *These steps are actionable, low-risk, and directly address gaps in the current draft. They ensure compliance and feasibility.*

--- 

You now have a clear, structured, and BFH-compliant proposal — grounded in what you know, honest about what you don’t, and ready for supervisor feedback.

<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    The refinement is well executed. Suggested changes are clearly highlighted, and the
    original structure is preserved. Applying the same refinement approach to
    <code>GPT-OSS 120B</code> would allow for a more consistent comparison between the two
    models.
  </div>
</div>


In [75]:
model_name = "GPT‑OSS 120B"
mode = "Proposal refinement assistant"
persona = "Helper"

system_prompt, temp = build_system_prompt(mode, persona)

answer = gpt_oss_completion(
    system_prompt=system_prompt,
    user_prompt=few_shot_user,
    temperature=0.2,
)

show_markdown(answer, f"{model_name} – Condition C (few-shot + draft refine)")



### GPT‑OSS 120B – Condition C (few-shot + draft refine)

**BFH‑Thesis Proposal – Revised Draft**  

---

### 1. Working Title  
**Using AI‑Assisted Writing in Bachelor Theses: Practices, Prompt Engineering, and Perceived Risks**

---

### 2. Introduction  
Artificial‑intelligence (AI) language models such as ChatGPT are now widely available and are increasingly used by students for academic writing. While these tools can support literature searches, drafting, and editing, they also generate inaccurate or fabricated information (“hallucinations”). At the Bern University of Applied Sciences (BFH) the extent to which students rely on AI for their bachelor theses, how they formulate prompts, and which risks they perceive has not been systematically examined. This study will fill that gap by providing an empirical overview of current AI‑usage patterns and by evaluating whether deliberate prompt‑engineering improves the quality and reliability of AI‑generated content.

---

### 3. Objective  
The primary objective is to **characterise the current use of AI writing tools among BFH bachelor‑thesis students** and to **assess the impact of structured prompt‑engineering on the accuracy and usefulness of the generated text**. A secondary objective is to identify the main perceived benefits and risks associated with AI‑assisted thesis work.

---

### 4. Theoretical Basics  
- **AI language models** – brief description of large‑scale transformer models (e.g., ChatGPT) and their typical functionalities (text generation, summarisation, Q&A).  
- **Prompt engineering** – definition and relevance for steering model output, including concepts such as prompt length, specificity, and iterative refinement.  
- **Academic integrity & hallucination risk** – overview of known issues (fabricated citations, factual errors) and existing guidelines for responsible AI use in higher education.  
- **Technology‑acceptance frameworks** – e.g., the Technology Acceptance Model (TAM) to contextualise students’ attitudes toward AI tools.

*(TODO: add concise citations for each theoretical component – locate relevant BFH‑approved sources or recent peer‑reviewed articles.)*

---

### 5. Research Design / Method  

| Component | Description |
|-----------|-------------|
| **Study type** | Mixed‑methods exploratory study (quantitative survey + qualitative prompt‑analysis). |
| **Population** | Bachelor‑thesis students at BFH (all faculties). |
| **Sampling** | Convenience sampling via university mailing list; aim for **[TODO: specify target sample size, e.g., 150 respondents]** to achieve sufficient statistical power. |
| **Data collection** | 1. **Online questionnaire** (demographics, AI‑tool usage frequency, perceived benefits/risks, self‑rated competence in prompt formulation). <br>2. **Prompt‑task experiment**: participants submit two prompts for a predefined thesis‑related writing task – one “ad‑hoc” prompt and one “engineered” prompt based on a short guideline we provide. |
| **Variables** | - Independent variable: Prompt type (ad‑hoc vs. engineered). <br>- Dependent variables: (a) Accuracy of AI‑generated text (rated by independent expert reviewers), (b) Perceived usefulness (self‑rating), (c) Time spent on task. |
| **Blinding** | Reviewers of AI‑generated outputs will be blind to the prompt type. |
| **Data analysis** | - Descriptive statistics for usage patterns. <br>- Paired‑sample t‑tests (or non‑parametric equivalent) to compare accuracy and usefulness between prompt types. <br>- Thematic analysis of open‑ended survey responses on risks and benefits. |
| **Ethics** | Informed consent obtained online; data anonymised; ethical approval to be requested from the BFH Ethics Committee. |

*(TODO: finalize the exact wording of the prompt‑engineering guideline and the expert‑rating rubric.)*

---

### 6. Expected Results  
- **Usage prevalence**: A majority of respondents will report having used AI tools at least once for their thesis.  
- **Prompt‑engineering effect**: Engineered prompts are expected to yield significantly higher accuracy scores and higher perceived usefulness than ad‑hoc prompts, while not increasing completion time.  
- **Risk perception**: Students will identify hallucinations and citation errors as the main risks, and will express a need for clearer institutional guidelines.

---

### 7. Outline (Provisional Chapter Structure)  

1. **Introduction** – context, problem statement, research gap.  
2. **Theoretical Foundations** – AI language models, prompt engineering, academic integrity.  
3. **Methodology** – design, participants, instruments, procedure, ethical considerations.  
4. **Results** – quantitative survey findings, prompt‑experiment outcomes, qualitative insights.  
5. **Discussion** – interpretation, implications for teaching, limitations, future research.  
6. **Conclusion** – summary of contributions and recommendations for BFH policy.  

*(TODO: confirm chapter titles with supervisor; adjust length to fit required page count.)*

---

### 8. Project Planning (Timeline)  

| Month | Milestone |
|-------|-----------|
| **Sept – Oct** | Literature review; finalize questionnaire and prompt‑engineering guideline. |
| **Nov** | Submit ethics application; pilot test survey with 5–10 students. |
| **Dec – Jan** | Data collection (online survey + prompt task). |
| **Feb** | Data cleaning; expert rating of AI outputs. |
| **Mar** | Statistical and thematic analysis. |
| **Apr** | Draft results and discussion chapters. |
| **May** | Write introduction, methods, and conclusion; integrate supervisor feedback. |
| **June** | Final proofreading, formatting, and submission. |

*(TODO: verify exact deadline dates with the BFH thesis office.)*

---

### 9. Literature (Preliminary List)  

- Overview of AI language models and their capabilities.  
- Studies on prompt engineering and its impact on output quality.  
- Research on AI use in higher‑education writing (e.g., student attitudes, plagiarism concerns).  
- BFH or Swiss higher‑education policies on AI and academic integrity.  

*(TODO: populate with full citations; ensure all sources are accessible through the university library.)*

---

## Next Steps (max 3 concrete actions)

1. **Define the sample size and recruit participants** – calculate the required number of respondents for the planned statistical tests and draft the invitation email to be sent via the BFH mailing list.  
2. **Develop the prompt‑engineering guideline and rating rubric** – create a one‑page prompt template (e.g., specifying context, task, constraints) and a 5‑point accuracy checklist for the expert reviewers.  
3. **Prepare the ethics application** – write the consent form, data‑management plan, and submit the protocol to the BFH Ethics Committee before the end of November.  

These actions will give you a solid foundation to complete the remaining sections of the proposal and keep the project on schedule. Good luck!



<div style="background:#eff6ff; border-left:5px solid #3b82f6; padding:12px 14px; border-radius:10px; margin:10px 0;">
  <div style="font-weight:700; color:#374151; margin-bottom:6px;">
    Observation
  </div>
  <div style="color:#374151;">
    In comparison to Qwen it is less clear what is from the user and what has been improved. It seems like GPT just imporved things implicitly. Which on one hand is ready to be used and on the other hand doesnt engage the user to think actively.
  </div>
</div>





## Conclusion

In this notebook we systematically compared different **prompting strategies** (baseline, engineered with safety and structure, personas, and few‑shot examples) across two **LLM**: Qwen3 Next 80B A3B and OpenAI GPT‑OSS 120B. For each condition we kept the user task constant and only changed the system prompt, mode/persona settings, and—where relevant—the retrieval setup and proposal examples. This made it possible to see how much of the behaviour comes from *prompt design* versus *model choice*.

The main design decisions that flowed into the current Proposify agent are:

- Use an explicit **RAG safety preamble** and task‑specific headings to reduce hallucinations and make answers easier for students to reuse.
- Represent the user’s intent at a high level via **mode** (“Research question helper” vs “Proposal refinement assistant”) and then tune tone and temperature via **personas** (“Supervisor”, “Helper”, “Creative”).
- For methods and gap questions, augment retrieval queries with small, fixed **tag strings** (“validity, reliability, sampling…”) so Chroma returns Creswell/BFH chunks that are actually useful for the task.
- In proposal refinement, combine the **BFH proposal template** (for section expectations) with **real proposal examples** from OSF (for depth and tone), while clearly instructing the model not to copy example text.

Overall, the experiments show that careful prompt engineering (safety rules, headings, modes, personas, retrieval tags, and few‑shot examples grounded in real proposals) has at least as much impact on usefulness and robustness as the specific model used. The current Proposify agent uses the patterns that worked best here as its default behaviour, while this notebook remains a sandbox for testing new prompts and future LLMs in a controlled way.
